# US Flight Delays and Cancellations 2015 — analyse complete

**Cours** : restitution de projet data · **Dataset** : vols domestiques americains 2015 (Kaggle / US DOT).

Ce notebook part **uniquement des fichiers bruts** `data/raw/flights.csv`, `data/raw/airlines.csv` et
`data/raw/airports.csv`. Il execute la chaine complete : chargement, audit qualite, preparation,
test des trois hypotheses, export des tableaux et generation des donnees du dashboard HTML.

## Les trois hypotheses testees

| # | Hypothese | Ce qu'on mesure |
|---|---|---|
| **H1** | Il y a plus de vols en retard pendant les vacances que le reste de l'annee. | Taux de vols en retard a l'arrivee, selon la periode de l'annee. |
| **H2** | Plus une compagnie fait de vols longue distance, plus ses vols sont en retard. | Correlation entre part de vols >= 1500 miles et taux de retard, sur les 14 compagnies. |
| **H3** | Une petite minorite de liaisons concentre la majorite des annulations. | Concentration des annulations par route, comparee a la concentration du trafic. |

## Mode d'emploi

1. Placer les trois CSV bruts dans `data/raw/`.
2. `Cell > Run All`. Duree indicative : 2 a 4 minutes (le CSV brut fait ~592 Mo).
3. Tous les livrables sont ecrits dans `v2/outputs/` et `v2/dashboard/dashboard_data.js`.

> **Regle suivie dans tout le notebook** : aucun chiffre n'est ecrit en dur dans le texte.
> Les commentaires et la synthese finale sont generes a partir des variables calculees,
> pour qu'une reexecution ne puisse jamais contredire le texte.

---
## 1. Environnement et chemins

On enregistre les versions des librairies pour que le resultat soit tracable, et on localise la racine
du projet en remontant l'arborescence jusqu'a trouver `data/raw`. Le notebook fonctionne donc quel que
soit le dossier depuis lequel Jupyter a ete lance.

In [1]:
import json
import math
import sys
import time
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
T0 = time.time()

# --- racine du projet : on remonte jusqu'au dossier qui contient data/raw ---
def trouver_racine(depart: Path) -> Path:
    for p in [depart, *depart.parents]:
        if (p / "data" / "raw" / "flights.csv").exists():
            return p
    raise FileNotFoundError(
        "Impossible de trouver 'data/raw/flights.csv' en remontant depuis " + str(depart)
        + ". Placez le notebook dans le projet ou corrigez RACINE manuellement."
    )

RACINE = trouver_racine(Path.cwd().resolve())
RAW = RACINE / "data" / "raw"
V2 = RACINE / "v2"
OUT = V2 / "outputs"
FIG = OUT / "figures"
DASH = V2 / "dashboard"
ASSETS = V2 / "assets"
for d in (OUT, FIG, DASH, ASSETS):
    d.mkdir(parents=True, exist_ok=True)

def nb_fr(n: float, dec: int = 0) -> str:
    """Formate un nombre a la francaise : 4 693 et non 4,693."""
    return f"{n:,.{dec}f}".replace(",", " ")

print("Python     ", sys.version.split()[0])
print("pandas     ", pd.__version__)
print("numpy      ", np.__version__)
print("scipy      ", scipy.__version__)
print("matplotlib ", matplotlib.__version__)
print()
print("Racine projet :", RACINE)
print("Donnees brutes:", RAW)
print("Sorties       :", OUT)

Python      3.11.9
pandas      3.0.3
numpy       2.4.6
scipy       1.17.1
matplotlib  3.11.0

Racine projet : C:\Users\Piks\OneDrive\Documents\ESGI\restitution\codex
Donnees brutes: C:\Users\Piks\OneDrive\Documents\ESGI\restitution\codex\data\raw
Sorties       : C:\Users\Piks\OneDrive\Documents\ESGI\restitution\codex\v2\outputs


---
## 2. Chargement des donnees brutes

`flights.csv` fait environ 592 Mo. On ne charge que les colonnes utiles et on impose des types compacts
(`int8`, `int16`, `float32`, `category`) : sans cela, pandas alloue des `int64`/`float64` partout et
l'empreinte memoire depasse 2 Go.

Les trois fichiers :

- **`flights.csv`** — une ligne par vol domestique programme en 2015.
- **`airlines.csv`** — correspondance code IATA compagnie -> nom commercial (14 lignes).
- **`airports.csv`** — 322 aeroports avec ville, etat et coordonnees GPS (utilisees pour la carte).

In [2]:
COLONNES = {
    # identification du vol
    "YEAR": "int16", "MONTH": "int8", "DAY": "int8", "DAY_OF_WEEK": "int8",
    "AIRLINE": "category", "ORIGIN_AIRPORT": "category", "DESTINATION_AIRPORT": "category",
    # horaires et distance
    "SCHEDULED_DEPARTURE": "int16", "DISTANCE": "int16",
    # resultats du vol
    "DEPARTURE_DELAY": "float32", "ARRIVAL_DELAY": "float32",
    "DIVERTED": "int8", "CANCELLED": "int8", "CANCELLATION_REASON": "category",
    # decomposition des minutes de retard
    "AIR_SYSTEM_DELAY": "float32", "SECURITY_DELAY": "float32", "AIRLINE_DELAY": "float32",
    "LATE_AIRCRAFT_DELAY": "float32", "WEATHER_DELAY": "float32",
}

t = time.time()
vols = pd.read_csv(RAW / "flights.csv", usecols=list(COLONNES), dtype=COLONNES)
compagnies_ref = pd.read_csv(RAW / "airlines.csv")
aeroports_ref = pd.read_csv(RAW / "airports.csv")

print(f"flights.csv  : {vols.shape[0]:>9,} lignes x {vols.shape[1]} colonnes  "
      f"({vols.memory_usage(deep=True).sum() / 1e6:.0f} Mo, {time.time() - t:.0f} s)")
print(f"airlines.csv : {compagnies_ref.shape[0]:>9,} lignes")
print(f"airports.csv : {aeroports_ref.shape[0]:>9,} lignes")

# Controles d'integrite : si l'un echoue, le fichier source n'est pas celui attendu.
assert len(vols) == 5_819_079, f"Nombre de vols inattendu : {len(vols):,}"
assert set(vols.YEAR.unique()) == {2015}, "Le fichier doit contenir uniquement l'annee 2015"
assert len(compagnies_ref) == 14, "14 compagnies attendues"
print("\nControles d'integrite : OK")

flights.csv  : 5,819,079 lignes x 19 colonnes  (262 Mo, 5 s)
airlines.csv :        14 lignes
airports.csv :       322 lignes

Controles d'integrite : OK


---
## 3. Audit qualite

Avant toute analyse, on regarde ce qui manque et ce qui est incoherent. Deux points sortent de cet audit
et conditionnent la suite : le traitement des valeurs manquantes de `ARRIVAL_DELAY`, et un probleme de
codage des aeroports qui touche un mois entier.

In [3]:
audit = pd.DataFrame({
    "type": vols.dtypes.astype(str),
    "manquants": vols.isna().sum(),
})
audit["manquants_pct"] = (audit.manquants / len(vols) * 100).round(2)
audit.sort_values("manquants", ascending=False)

,type,manquants,manquants_pct
CANCELLATION_REASON,category,5729195,98.46
SECURITY_DELAY,float32,4755640,81.72
WEATHER_DELAY,float32,4755640,81.72
AIR_SYSTEM_DELAY,float32,4755640,81.72
LATE_AIRCRAFT_DELAY,float32,4755640,81.72
AIRLINE_DELAY,float32,4755640,81.72
ARRIVAL_DELAY,float32,105071,1.81
DEPARTURE_DELAY,float32,86153,1.48
DAY,int8,0,0.00
YEAR,int16,0,0.00


### 3.1 Pourquoi `ARRIVAL_DELAY` est manquant

Les 105 071 valeurs manquantes de `ARRIVAL_DELAY` ne sont pas du bruit : ce sont exactement les vols qui
**n'ont jamais atterri comme prevu**, c'est-a-dire les vols annules et les vols deroutes. C'est structurel,
pas accidentel.

Consequence directe : ecrire `ARRIVAL_DELAY >= 15` sur le dataset complet renvoie `False` pour ces vols,
donc **les compte silencieusement comme des vols a l'heure**. On les exclut explicitement du perimetre des
retards (variable `EST_EXPLOITABLE` construite en section 4).

In [4]:
manque_arr = vols.ARRIVAL_DELAY.isna()
verif = pd.crosstab(
    np.select([vols.CANCELLED.eq(1), vols.DIVERTED.eq(1)], ["Annule", "Deroute"], default="Vol effectue"),
    np.where(manque_arr, "ARRIVAL_DELAY manquant", "ARRIVAL_DELAY renseigne"),
)
print(verif.to_string())

restant = int(manque_arr.sum() - vols.CANCELLED.sum() - vols.DIVERTED.sum())
print(f"\nAnnules  : {int(vols.CANCELLED.sum()):>7,}")
print(f"Deroutes : {int(vols.DIVERTED.sum()):>7,}")
print(f"Manquants non expliques par ces deux cas : {restant:,}")

col_0         ARRIVAL_DELAY manquant  ARRIVAL_DELAY renseigne
row_0                                                        
Annule                         89884                        0
Deroute                        15187                        0
Vol effectue                       0                  5714008

Annules  :  89,884
Deroutes :  15,187
Manquants non expliques par ces deux cas : 0


### 3.2 Le probleme des codes aeroport numeriques

`ORIGIN_AIRPORT` et `DESTINATION_AIRPORT` contiennent normalement des codes IATA a 3 lettres (`LAX`, `ORD`...).
Mais une partie des lignes utilise des **identifiants numeriques DOT** (`10397`, `13930`...), qui ne sont pas
joignables a `airports.csv`.

La cellule suivante montre que ces lignes ne sont pas reparties au hasard.

In [5]:
orig = vols.ORIGIN_AIRPORT.astype("string")
dest = vols.DESTINATION_AIRPORT.astype("string")
code_numerique = orig.str.fullmatch(r"\d+").fillna(False) | dest.str.fullmatch(r"\d+").fillna(False)

repartition = (
    vols.loc[code_numerique].groupby("MONTH", observed=True).size()
    .rename("lignes en code numerique").to_frame()
)
repartition["part du mois (%)"] = (
    repartition["lignes en code numerique"] / vols.groupby("MONTH", observed=True).size() * 100
).round(1)

print(f"Lignes concernees : {int(code_numerique.sum()):,} soit {code_numerique.mean() * 100:.2f} % du dataset")
print(f"Mois concernes    : {sorted(vols.loc[code_numerique, 'MONTH'].unique().tolist())}")
print()
print(repartition.to_string())

Lignes concernees : 486,165 soit 8.35 % du dataset
Mois concernes    : [10]

       lignes en code numerique  part du mois (%)
MONTH                                            
10                       486165             100.0


In [6]:
# Que represente octobre dans le phenomene etudie par H3 ?
annul_oct = int(vols.loc[vols.MONTH.eq(10), "CANCELLED"].sum())
annul_tot = int(vols.CANCELLED.sum())
taux_oct = vols.loc[vols.MONTH.eq(10), "CANCELLED"].mean() * 100
taux_an = vols.CANCELLED.mean() * 100

print(f"Octobre = {code_numerique.mean() * 100:.2f} % des vols de l'annee")
print(f"Octobre = {annul_oct:,} annulations sur {annul_tot:,}, soit {annul_oct / annul_tot * 100:.2f} % du total")
print(f"Taux d'annulation en octobre : {taux_oct:.2f} %  (moyenne annuelle : {taux_an:.2f} %)")

Octobre = 8.35 % des vols de l'annee
Octobre = 2,454 annulations sur 89,884, soit 2.73 % du total
Taux d'annulation en octobre : 0.50 %  (moyenne annuelle : 1.54 %)


**Decision et consequence assumee.** La totalite des codes numeriques est concentree sur **octobre 2015** :
ce mois utilise un autre referentiel d'aeroports que les onze autres. Comme H3 raisonne route par route,
on ne peut pas melanger deux referentiels — une meme liaison apparaitrait deux fois sous deux noms differents.

On restreint donc **H3 aux onze mois en codes IATA**. Filtrer sur `code IATA lisible` revient donc
exactement a **exclure octobre**, ce qui doit etre dit explicitement plutot que subi.

Ce que ca coute : octobre pese 8,35 % des vols mais seulement 2,73 % des annulations, avec un taux
d'annulation de 0,50 % contre 1,54 % sur l'annee. C'est **le mois le plus calme de 2015**. Son exclusion
retire donc peu du phenomene etudie, et va plutot dans le sens d'une sous-estimation de la concentration.

**H1 et H2 conservent les douze mois** : elles raisonnent sur des dates et des compagnies, pas sur des routes,
et ne sont donc pas affectees par ce probleme de codage.

---
## 4. Preparation des donnees

On construit les variables d'analyse. Chaque variable derivee repond a un besoin precis des hypotheses ;
elles sont recapitulees dans le dictionnaire en fin de section.

In [7]:
vols["DATE"] = pd.to_datetime(dict(year=vols.YEAR, month=vols.MONTH, day=vols.DAY), errors="coerce")
vols["JOUR_ANNEE"] = vols.DATE.dt.dayofyear.astype("int16")
vols["HEURE_DEP"] = ((vols.SCHEDULED_DEPARTURE // 100) % 24).astype("int8")

# Perimetre des retards : un vol annule ou deroute n'a pas de retard d'arrivee mesurable.
vols["EST_EXPLOITABLE"] = vols.CANCELLED.eq(0) & vols.DIVERTED.eq(0)

# Definition du retard, alignee sur le standard US DOT : 15 minutes ou plus a l'arrivee.
SEUIL_RETARD = 15
vols["EST_EN_RETARD"] = vols.ARRIVAL_DELAY.ge(SEUIL_RETARD) & vols.EST_EXPLOITABLE

# Vol long-courrier domestique (H2) : il n'y a pas de long-courrier au sens international dans ce
# dataset 100 % domestique. On prend 1500 miles, seuil au-dela duquel un vol traverse le pays.
SEUIL_LONG = 1500
vols["EST_VOL_LONG"] = vols.DISTANCE.ge(SEUIL_LONG)

# Route et perimetre exploitable pour H3 (cf. section 3.2).
vols["ROUTE_LISIBLE"] = ~code_numerique
vols["ROUTE"] = orig.where(vols.ROUTE_LISIBLE) + " -> " + dest.where(vols.ROUTE_LISIBLE)

assert vols.DATE.isna().sum() == 0, "Des dates n'ont pas pu etre construites"
assert vols.loc[vols.EST_EXPLOITABLE, "ARRIVAL_DELAY"].isna().sum() == 0, \
    "Il reste des retards manquants dans le perimetre exploitable"
print("Variables construites, controles OK.")

Variables construites, controles OK.


### 4.1 Fenetres de forte mobilite (variable centrale de H1)

Le dataset ne contient **aucune colonne vacances** : il faut la fabriquer. On definit quatre fenetres de
forte mobilite du calendrier americain, chacune bornee explicitement pour etre reproductible et discutable :

| Fenetre | Dates 2015 | Nature |
|---|---|---|
| Vacances d'ete | 1er juin -> 31 aout | saison longue, hausse durable du trafic loisir |
| Nouvel an | 1er -> 5 janvier | pic court, retours de fetes |
| Thanksgiving | 20 -> 30 novembre | pic court, plus gros week-end de deplacement US |
| Noel / fin d'annee | 18 -> 31 decembre | pic court, departs de fetes |

C'est un **proxy**, pas une verite : il sera confronte a la realite des donnees en section 6.

In [8]:
d = vols.DATE
FENETRES = {
    "Vacances d'ete":     (vols.MONTH.isin([6, 7, 8]),                    "juin -> aout"),
    "Nouvel an":          (vols.MONTH.eq(1) & vols.DAY.le(5),             "1 -> 5 janvier"),
    "Thanksgiving":       (d.between("2015-11-20", "2015-11-30"),         "20 -> 30 novembre"),
    "Noel / fin d'annee": (d.between("2015-12-18", "2015-12-31"),         "18 -> 31 decembre"),
}

vols["EST_VACANCES"] = np.logical_or.reduce([m for m, _ in FENETRES.values()])
vols["FENETRE"] = np.select(
    [m for m, _ in FENETRES.values()], list(FENETRES), default="Hors vacances",
)
# Pics de fin d'annee seuls : utile pour isoler l'effet saison de l'effet fete.
vols["EST_PIC_FETE"] = np.logical_or.reduce([FENETRES[k][0] for k in
                                             ("Nouvel an", "Thanksgiving", "Noel / fin d'annee")])

print(vols.FENETRE.value_counts().to_string())

FENETRE
Hors vacances         3823210
Vacances d'ete        1535151
Noel / fin d'annee     216004
Thanksgiving           165689
Nouvel an               79025


In [9]:
DICTIONNAIRE = pd.DataFrame([
    ("DATE",            "date",    "Date du vol, reconstruite depuis YEAR/MONTH/DAY"),
    ("EST_EXPLOITABLE", "bool",    "Vol ni annule ni deroute : seul perimetre ou le retard existe"),
    ("EST_EN_RETARD",   "bool",    f"Arrivee avec >= {SEUIL_RETARD} min de retard (standard US DOT), sur vols exploitables"),
    ("EST_VOL_LONG",    "bool",    f"Distance >= {SEUIL_LONG} miles : long-courrier domestique (H2)"),
    ("EST_VACANCES",    "bool",    "Vol dans l'une des 4 fenetres de forte mobilite (H1)"),
    ("FENETRE",         "texte",   "Nom de la fenetre de mobilite, ou 'Hors vacances'"),
    ("EST_PIC_FETE",    "bool",    "Fenetre courte de fete uniquement (hors ete), pour isoler l'effet fete"),
    ("ROUTE",           "texte",   "ORIGINE -> DESTINATION, uniquement si les deux codes sont IATA (H3)"),
    ("ROUTE_LISIBLE",   "bool",    "Les deux aeroports ont un code IATA exploitable (exclut octobre)"),
    ("HEURE_DEP",       "int",     "Heure de depart programmee, 0 a 23"),
], columns=["variable", "type", "definition"])
DICTIONNAIRE

,variable,type,definition
0,DATE,date,"Date du vol, reconstruite depuis YEAR/MONTH/DAY"
1,EST_EXPLOITABLE,bool,Vol ni annule ni deroute : seul perimetre ou l...
2,EST_EN_RETARD,bool,Arrivee avec >= 15 min de retard (standard US ...
3,EST_VOL_LONG,bool,Distance >= 1500 miles : long-courrier domesti...
4,EST_VACANCES,bool,Vol dans l'une des 4 fenetres de forte mobilit...
5,FENETRE,texte,"Nom de la fenetre de mobilite, ou 'Hors vacances'"
6,EST_PIC_FETE,bool,"Fenetre courte de fete uniquement (hors ete), ..."
7,ROUTE,texte,"ORIGINE -> DESTINATION, uniquement si les deux..."
8,ROUTE_LISIBLE,bool,Les deux aeroports ont un code IATA exploitabl...
9,HEURE_DEP,int,"Heure de depart programmee, 0 a 23"


---
## 5. Panorama general

Avant de tester les hypotheses, on pose les ordres de grandeur : ce sont les reperes auxquels tous les
resultats suivants seront compares.

In [10]:
exploitables = vols[vols.EST_EXPLOITABLE]

TAUX_RETARD_GLOBAL = exploitables.EST_EN_RETARD.mean() * 100
TAUX_ANNUL_GLOBAL = vols.CANCELLED.mean() * 100
RETARD_MOYEN_GLOBAL = exploitables.ARRIVAL_DELAY.mean()

kpi = {
    "vols_programmes": len(vols),
    "vols_exploitables": int(vols.EST_EXPLOITABLE.sum()),
    "vols_annules": int(vols.CANCELLED.sum()),
    "vols_deroutes": int(vols.DIVERTED.sum()),
    "compagnies": int(vols.AIRLINE.nunique()),
    # Uniquement les codes IATA : les identifiants numeriques d'octobre gonfleraient artificiellement
    # ce compte en faisant apparaitre chaque aeroport une seconde fois sous un autre code.
    "aeroports": int(pd.concat([orig[vols.ROUTE_LISIBLE], dest[vols.ROUTE_LISIBLE]]).nunique()),
    "routes_lisibles": int(vols.loc[vols.ROUTE_LISIBLE, "ROUTE"].nunique()),
    "taux_retard_pct": round(TAUX_RETARD_GLOBAL, 2),
    "taux_annulation_pct": round(TAUX_ANNUL_GLOBAL, 2),
    "retard_moyen_min": round(float(RETARD_MOYEN_GLOBAL), 2),
    "retard_median_min": round(float(exploitables.ARRIVAL_DELAY.median()), 2),
}
for k, v in kpi.items():
    print(f"{k:22s} {v:>12,}" if isinstance(v, int) else f"{k:22s} {v:>12}")

vols_programmes           5,819,079
vols_exploitables         5,714,008
vols_annules                 89,884
vols_deroutes                15,187
compagnies                       14
aeroports                       322
routes_lisibles               4,693
taux_retard_pct               18.61
taux_annulation_pct            1.54
retard_moyen_min               4.41
retard_median_min              -5.0


Premier constat a garder en tete : le retard **median** est negatif alors que le retard **moyen** est positif.
La majorite des vols arrivent en avance, et la moyenne est tiree vers le haut par une minorite de vols
tres en retard. C'est pourquoi tout le notebook raisonne sur un **taux de vols en retard** (part de vols
au-dela de 15 minutes) plutot que sur une moyenne, qui serait trompeuse.

In [11]:
# Series temporelles : mensuelle (lecture) et journaliere (detection des pics)
mensuel = vols.groupby("MONTH", observed=True).agg(
    vols=("CANCELLED", "size"), taux_annulation=("CANCELLED", "mean")).reset_index()
mensuel = mensuel.merge(
    exploitables.groupby("MONTH", observed=True).EST_EN_RETARD.mean().rename("taux_retard").reset_index(),
    on="MONTH")
mensuel["taux_annulation"] *= 100
mensuel["taux_retard"] *= 100

journalier = vols.groupby("DATE", observed=True).agg(
    vols=("CANCELLED", "size"), taux_annulation=("CANCELLED", "mean")).reset_index()
journalier = journalier.merge(
    exploitables.groupby("DATE", observed=True).EST_EN_RETARD.mean().rename("taux_retard").reset_index(),
    on="DATE")
journalier["taux_annulation"] *= 100
journalier["taux_retard"] *= 100
journalier["jour_annee"] = journalier.DATE.dt.dayofyear
journalier["fenetre"] = vols.groupby("DATE", observed=True).FENETRE.first().values

print(mensuel.round(2).to_string(index=False))
print(f"\nJours couverts : {len(journalier)}")

 MONTH   vols  taux_annulation  taux_retard
     1 469968             2.55        21.00
     2 429191             4.78        23.35
     3 504312             2.18        19.40
     4 485151             0.93        17.16
     5 496993             1.15        18.31
     6 503897             1.81        23.48
     7 520718             0.92        20.92
     8 510536             0.99        18.67
     9 464946             0.45        13.00
    10 486165             0.50        12.44
    11 467972             0.98        15.26
    12 479230             1.68        20.60

Jours couverts : 365


In [12]:
PIRES_RETARDS = journalier.nlargest(5, "taux_retard")[["DATE", "vols", "taux_retard", "taux_annulation"]]
PIRES_ANNULS = journalier.nlargest(5, "taux_annulation")[["DATE", "vols", "taux_retard", "taux_annulation"]]
print("Les 5 pires journees en retards :")
print(PIRES_RETARDS.round(2).to_string(index=False))
print("\nLes 5 pires journees en annulations :")
print(PIRES_ANNULS.round(2).to_string(index=False))

Les 5 pires journees en retards :
      DATE  vols  taux_retard  taux_annulation
2015-01-04 16352        49.30             2.65
2015-01-03 15434        45.22             2.14
2015-12-30 16260        43.65             1.73
2015-03-01 15171        41.31             9.98
2015-12-29 16199        39.92             4.23

Les 5 pires journees en annulations :
      DATE  vols  taux_retard  taux_annulation
2015-01-27 15155         7.13            19.03
2015-02-02 15975        28.88            17.52
2015-03-05 16622        31.21            17.19
2015-02-01 13406        21.83            14.76
2015-12-28 16312        36.38            13.35


C:\Users\Piks\AppData\Local\Temp\ipykernel_24232\388110008.py:4: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(PIRES_RETARDS.round(2).to_string(index=False))
C:\Users\Piks\AppData\Local\Temp\ipykernel_24232\388110008.py:6: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(PIRES_ANNULS.round(2).to_string(index=False))


Ces deux classements ne se recoupent pas, et c'est une information a part entiere :

- Les pires journees de **retard** sont des journees de **retour de fetes** (debut janvier, fin decembre).
- Les pires journees d'**annulation** sont des journees de **tempete hivernale** (fin janvier, fevrier, debut mars).

Retard et annulation ne sont donc pas le meme phenomene, et ne repondent pas aux memes causes. C'est la
grille de lecture qui structure H1 (retards, effet calendaire) et H3 (annulations, effet meteo/reseau).

In [13]:
# D'ou viennent les minutes de retard, et pourquoi annule-t-on un vol ?
COLS_CAUSES = ["LATE_AIRCRAFT_DELAY", "AIRLINE_DELAY", "AIR_SYSTEM_DELAY", "WEATHER_DELAY", "SECURITY_DELAY"]
LIB_CAUSES = {
    "LATE_AIRCRAFT_DELAY": "Avion arrive en retard", "AIRLINE_DELAY": "Compagnie",
    "AIR_SYSTEM_DELAY": "Controle aerien / aeroport", "WEATHER_DELAY": "Meteo", "SECURITY_DELAY": "Securite",
}
minutes = exploitables.loc[exploitables.EST_EN_RETARD, COLS_CAUSES].sum().astype("float64")
causes_retard = (minutes / minutes.sum() * 100).round(2).rename("part_minutes_pct").reset_index()
causes_retard["cause"] = causes_retard["index"].map(LIB_CAUSES)

LIB_ANNUL = {"A": "Compagnie", "B": "Meteo", "C": "Controle aerien / aeroport", "D": "Securite"}
causes_annul = (vols.loc[vols.CANCELLED.eq(1), "CANCELLATION_REASON"]
                .value_counts().rename("vols").reset_index())
causes_annul.columns = ["code", "vols"]
causes_annul["cause"] = causes_annul.code.map(LIB_ANNUL)
causes_annul["part_pct"] = causes_annul.vols / causes_annul.vols.sum() * 100

print("Repartition des minutes de retard :")
print(causes_retard[["cause", "part_minutes_pct"]].round(2).to_string(index=False))
print("\nMotifs d'annulation :")
print(causes_annul[["cause", "vols", "part_pct"]].round(2).to_string(index=False))

Repartition des minutes de retard :
                     cause  part_minutes_pct
    Avion arrive en retard             39.84
                 Compagnie             32.20
Controle aerien / aeroport             22.88
                     Meteo              4.95
                  Securite              0.13

Motifs d'annulation :
                     cause  vols  part_pct
                     Meteo 48851     54.35
                 Compagnie 25262     28.11
Controle aerien / aeroport 15749     17.52
                  Securite    22      0.02


---
# 6. H1 — « Il y a plus de vols en retard pendant les vacances que le reste de l'annee »

On procede en trois temps volontairement separes :

1. **Le test naif** — celui qu'on ferait spontanement : vacances contre hors vacances.
2. **La decomposition** — qui montre que l'agregat cache des comportements opposes.
3. **Le test controle** — qui compare chaque fenetre a une reference comparable, et donne la vraie reponse.

### 6.1 Test naif : vacances contre hors vacances

In [14]:
h1_naif = exploitables.groupby(vols.EST_VACANCES, observed=True).agg(
    vols=("EST_EN_RETARD", "size"),
    taux_retard=("EST_EN_RETARD", "mean"),
    retard_moyen=("ARRIVAL_DELAY", "mean"),
    retard_median=("ARRIVAL_DELAY", "median"),
).reset_index()
h1_naif["periode"] = np.where(h1_naif.EST_VACANCES, "Periodes de vacances", "Hors vacances")
h1_naif["taux_retard"] *= 100

TAUX_VAC = float(h1_naif.loc[h1_naif.EST_VACANCES, "taux_retard"].iloc[0])
TAUX_HORS = float(h1_naif.loc[~h1_naif.EST_VACANCES, "taux_retard"].iloc[0])
ECART_NAIF = TAUX_VAC - TAUX_HORS

h1_naif[["periode", "vols", "taux_retard", "retard_moyen", "retard_median"]].round(2)

,periode,vols,taux_retard,retard_moyen,retard_median
0,Hors vacances,3753092,16.89,2.74,-5.0
1,Periodes de vacances,1960916,21.91,7.60,-4.0


In [15]:
def test_chi2(masque_groupe, masque_cible, base):
    """Chi-2 d'independance + taille d'effet. Renvoie un dict lisible.

    Sur plusieurs millions de lignes, la p-value n'apprend plus rien : elle est toujours
    inferieure a 0,05. On renvoie donc systematiquement le V de Cramer et le risque relatif,
    qui eux mesurent l'intensite de la relation et non seulement son existence.
    """
    table = pd.crosstab(masque_groupe.loc[base.index], masque_cible.loc[base.index])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    n = table.values.sum()
    v = math.sqrt(chi2 / (n * (min(table.shape) - 1)))
    p1 = table.iloc[1, 1] / table.iloc[1].sum()
    p0 = table.iloc[0, 1] / table.iloc[0].sum()
    return {"chi2": round(float(chi2), 1), "p_value": float(p), "dof": int(dof),
            "cramer_v": round(v, 4), "risque_relatif": round(float(p1 / p0), 3), "n": int(n)}

H1_TEST = test_chi2(vols.EST_VACANCES, vols.EST_EN_RETARD, exploitables)

print(f"Taux en periode de vacances : {TAUX_VAC:.2f} %")
print(f"Taux hors vacances          : {TAUX_HORS:.2f} %")
print(f"Ecart                       : {ECART_NAIF:+.2f} points")
print(f"Risque relatif              : x{H1_TEST['risque_relatif']}")
print()
print(f"chi2 = {H1_TEST['chi2']:,.1f}   p = {H1_TEST['p_value']:.3g}   V de Cramer = {H1_TEST['cramer_v']}")
print(f"Interpretation du V : {'effet faible' if H1_TEST['cramer_v'] < 0.1 else 'effet modere'} "
      f"(seuils usuels : 0,1 faible / 0,3 modere / 0,5 fort)")

Taux en periode de vacances : 21.91 %
Taux hors vacances          : 16.89 %
Ecart                       : +5.03 points
Risque relatif              : x1.298

chi2 = 21,472.5   p = 0   V de Cramer = 0.0613
Interpretation du V : effet faible (seuils usuels : 0,1 faible / 0,3 modere / 0,5 fort)


**Ce que dit — et ne dit pas — ce premier resultat.** L'ecart est reel et va dans le sens de l'hypothese.
Mais la p-value ne doit convaincre personne ici : avec 5,7 millions d'observations, **n'importe quel ecart,
meme minuscule, ressort comme significatif**. Le V de Cramer, lui, indique une association faible.

Surtout, l'agregat « vacances » melange quatre fenetres tres differentes. On les separe avant de conclure.

### 6.2 Decomposition : l'agregat cache des comportements opposes

In [16]:
lignes = []
for nom, (masque, libelle) in FENETRES.items():
    sous = exploitables[masque.loc[exploitables.index]]
    lignes.append({
        "fenetre": nom, "dates": libelle, "vols": len(sous),
        "taux_retard": sous.EST_EN_RETARD.mean() * 100,
        "retard_moyen": sous.ARRIVAL_DELAY.mean(),
    })
hors = exploitables[~vols.EST_VACANCES.loc[exploitables.index]]
lignes.append({"fenetre": "Hors vacances", "dates": "reste de l'annee", "vols": len(hors),
               "taux_retard": hors.EST_EN_RETARD.mean() * 100, "retard_moyen": hors.ARRIVAL_DELAY.mean()})

h1_detail = pd.DataFrame(lignes).sort_values("taux_retard", ascending=False)
h1_detail["ecart_vs_hors_vacances"] = h1_detail.taux_retard - TAUX_HORS
h1_detail["part_du_volume_vacances_pct"] = np.where(
    h1_detail.fenetre.eq("Hors vacances"), np.nan,
    h1_detail.vols / h1_detail.loc[h1_detail.fenetre.ne("Hors vacances"), "vols"].sum() * 100)
h1_detail.round(2).to_string(index=False)

"           fenetre             dates    vols  taux_retard  retard_moyen  ecart_vs_hors_vacances  part_du_volume_vacances_pct\n         Nouvel an    1 -> 5 janvier   76931        36.17     18.559999                   19.28                         3.92\nNoel / fin d'annee 18 -> 31 decembre  209038        28.82     14.320000                   11.93                        10.66\n    Vacances d'ete      juin -> aout 1511187        21.01      6.860000                    4.12                        77.07\n     Hors vacances  reste de l'annee 3753092        16.89      2.740000                    0.00                          NaN\n      Thanksgiving 20 -> 30 novembre  163760        14.73      0.680000                   -2.16                         8.35"

In [17]:
POIDS_ETE = float(h1_detail.loc[h1_detail.fenetre.eq("Vacances d'ete"), "part_du_volume_vacances_pct"].iloc[0])
TAUX_THKS = float(h1_detail.loc[h1_detail.fenetre.eq("Thanksgiving"), "taux_retard"].iloc[0])

print(f"L'ete represente {POIDS_ETE:.1f} % du volume classe 'vacances'.")
print(f"L'agregat 'vacances' est donc a {POIDS_ETE:.0f} % un agregat 'ete'.")
print()
print(f"Thanksgiving : {TAUX_THKS:.2f} % de retards, soit {TAUX_THKS - TAUX_HORS:+.2f} points "
      f"vs hors vacances -> la fenetre est {'SOUS' if TAUX_THKS < TAUX_HORS else 'au-dessus de'} la reference.")

L'ete represente 77.1 % du volume classe 'vacances'.
L'agregat 'vacances' est donc a 77 % un agregat 'ete'.

Thanksgiving : 14.73 % de retards, soit -2.16 points vs hors vacances -> la fenetre est SOUS la reference.


**Le probleme est maintenant visible.** Thanksgiving — le plus gros week-end de deplacement des Etats-Unis —
affiche **moins** de retards que le reste de l'annee. Une fenetre sur quatre contredit donc frontalement
l'hypothese, et l'agregat le masque parce qu'il est domine aux trois quarts par l'ete.

Il y a en plus un biais de comparaison : comparer decembre a la moyenne annuelle, c'est comparer un mois
d'hiver a une reference qui contient le printemps et l'automne. On mesure alors la **saison**, pas les vacances.

### 6.3 Test controle : chaque fenetre contre une reference comparable

Correction du biais : chaque fenetre courte est comparee **au reste de son propre mois**, ce qui neutralise
la saison et la meteo moyenne du mois. L'ete, qui couvre des mois entiers, ne peut pas etre traite ainsi :
on le compare au reste de l'annee hors fenetres de fete, ce qui en fait une mesure d'effet saisonnier.

In [18]:
def effet_net(nom, masque):
    """Compare une fenetre a une reference comparable et renvoie l'ecart net en points."""
    if nom == "Vacances d'ete":
        ref_masque = ~vols.MONTH.isin([6, 7, 8]) & ~vols.EST_PIC_FETE
        ref_libelle = "reste de l'annee hors fetes"
        nature = "effet saisonnier"
    else:
        mois = sorted(vols.loc[masque, "MONTH"].unique().tolist())
        ref_masque = vols.MONTH.isin(mois) & ~masque
        ref_libelle = "reste du meme mois"
        nature = "effet fenetre"

    dans = exploitables[masque.loc[exploitables.index]]
    ref = exploitables[ref_masque.loc[exploitables.index]]
    base = exploitables[(masque | ref_masque).loc[exploitables.index]]
    test = test_chi2(masque, vols.EST_EN_RETARD, base)
    return {
        "fenetre": nom, "nature": nature, "reference": ref_libelle,
        "vols": len(dans), "taux_fenetre": dans.EST_EN_RETARD.mean() * 100,
        "taux_reference": ref.EST_EN_RETARD.mean() * 100,
        "effet_net_points": dans.EST_EN_RETARD.mean() * 100 - ref.EST_EN_RETARD.mean() * 100,
        "risque_relatif": test["risque_relatif"], "cramer_v": test["cramer_v"], "p_value": test["p_value"],
    }

h1_controle = pd.DataFrame([effet_net(n, m) for n, (m, _) in FENETRES.items()])
h1_controle = h1_controle.sort_values("effet_net_points", ascending=False).reset_index(drop=True)
h1_controle.round(3).to_string(index=False)

"           fenetre           nature                   reference    vols  taux_fenetre  taux_reference  effet_net_points  risque_relatif  cramer_v  p_value\n         Nouvel an    effet fenetre          reste du meme mois   76931        36.171          17.924            18.248           2.018     0.168      0.0\nNoel / fin d'annee    effet fenetre          reste du meme mois  209038        28.817          14.015            14.803           2.056     0.182      0.0\n    Vacances d'ete effet saisonnier reste de l'annee hors fetes 1511187        21.009          16.887             4.122           1.244     0.048      0.0\n      Thanksgiving    effet fenetre          reste du meme mois  163760        14.731          15.555            -0.823           0.947     0.011      0.0"

In [19]:
for _, r in h1_controle.iterrows():
    sens = "AUGMENTE" if r.effet_net_points > 0.5 else ("DIMINUE" if r.effet_net_points < -0.5 else "n'a pas d'effet sur")
    print(f"{r.fenetre:20s} {r.taux_fenetre:5.2f} % vs {r.taux_reference:5.2f} % ({r.reference:28s})"
          f" -> {r.effet_net_points:+6.2f} pt  : {sens} les retards")

Nouvel an            36.17 % vs 17.92 % (reste du meme mois          ) -> +18.25 pt  : AUGMENTE les retards
Noel / fin d'annee   28.82 % vs 14.01 % (reste du meme mois          ) -> +14.80 pt  : AUGMENTE les retards
Vacances d'ete       21.01 % vs 16.89 % (reste de l'annee hors fetes ) ->  +4.12 pt  : AUGMENTE les retards
Thanksgiving         14.73 % vs 15.55 % (reste du meme mois          ) ->  -0.82 pt  : DIMINUE les retards


### 6.4 Conclusion H1

In [20]:
fetes = h1_controle[h1_controle.nature.eq("effet fenetre")]
fetes_positives = fetes[fetes.effet_net_points > 0.5]
fetes_negatives = fetes[fetes.effet_net_points < -0.5]
effet_ete = float(h1_controle.loc[h1_controle.fenetre.eq("Vacances d'ete"), "effet_net_points"].iloc[0])
effet_max = h1_controle.iloc[0]

H1_STATUT = "Confirmee, mais a preciser"
H1_RESUME = (
    f"L'ecart brut est de {ECART_NAIF:+.2f} points ({TAUX_VAC:.2f} % contre {TAUX_HORS:.2f} %), "
    f"mais cet agregat est domine a {POIDS_ETE:.0f} % par l'ete et masque des comportements opposes. "
    f"Une fois chaque fenetre comparee a une reference comparable, l'effet le plus fort est "
    f"{effet_max.fenetre} ({effet_max.effet_net_points:+.2f} points vs {effet_max.reference}), "
    f"l'ete ne pese que {effet_ete:+.2f} points, et Thanksgiving est "
    f"{'negatif' if TAUX_THKS < TAUX_HORS else 'positif'} "
    f"({float(fetes.loc[fetes.fenetre.eq('Thanksgiving'), 'effet_net_points'].iloc[0]):+.2f} point). "
    f"Ce ne sont donc pas 'les vacances' qui creent du retard, mais les pics courts de fin d'annee."
)
print(H1_STATUT.upper()); print(); print(H1_RESUME)

CONFIRMEE, MAIS A PRECISER

L'ecart brut est de +5.03 points (21.91 % contre 16.89 %), mais cet agregat est domine a 77 % par l'ete et masque des comportements opposes. Une fois chaque fenetre comparee a une reference comparable, l'effet le plus fort est Nouvel an (+18.25 points vs reste du meme mois), l'ete ne pese que +4.12 points, et Thanksgiving est negatif (-0.82 point). Ce ne sont donc pas 'les vacances' qui creent du retard, mais les pics courts de fin d'annee.


**Reponse a l'hypothese : confirmee, mais pas pour la raison qu'elle suggere.**

L'effet existe et il est important, mais il est porte par les **pics courts de fin d'annee** (Nouvel an, Noel),
pas par « les vacances » en general. L'ete produit un effet modere, et Thanksgiving n'en produit aucun.

L'explication tient a la nature des pics : Noel et Nouvel an cumulent un trafic record **et** la meteo
hivernale, sur un reseau deja sature. Thanksgiving concentre son trafic sur peu de jours mais beneficie
d'une meteo de novembre plus clemente, et les compagnies y programment des marges horaires renforcees.

**Ce que ca implique pour le dashboard** : afficher un simple « vacances vs hors vacances » serait trompeur.
Il faut montrer la courbe journaliere, ou l'on voit les pics reels, et l'effet net fenetre par fenetre.

---
# 7. H2 — « Plus une compagnie fait de vols longue distance, plus ses vols sont en retard »

Le dataset est integralement domestique : il n'y a pas de long-courrier au sens international. On retient
**1500 miles ou plus**, seuil au-dela duquel un vol traverse une bonne partie du pays (Boston-Denver,
Chicago-Los Angeles).

Point de methode important : l'hypothese porte sur des **compagnies**, pas sur des vols. L'unite d'analyse
est donc la compagnie — il y en a 14. Tester cette hypothese sur 5,7 millions de vols reviendrait a compter
chaque vol comme une observation independante alors que tous les vols d'une meme compagnie partagent son
reseau, sa flotte et ses regles d'exploitation. C'est de la **pseudo-replication**, et ca fabrique de la
significativite artificielle.

In [21]:
noms = dict(zip(compagnies_ref.IATA_CODE, compagnies_ref.AIRLINE))

h2_compagnies = exploitables.groupby("AIRLINE", observed=True).agg(
    vols=("DISTANCE", "size"),
    vols_longs=("EST_VOL_LONG", "sum"),
    part_vols_longs=("EST_VOL_LONG", "mean"),
    taux_retard=("EST_EN_RETARD", "mean"),
    retard_moyen=("ARRIVAL_DELAY", "mean"),
    distance_moyenne=("DISTANCE", "mean"),
).reset_index()
h2_compagnies["compagnie"] = h2_compagnies.AIRLINE.map(noms)
h2_compagnies["part_vols_longs"] *= 100
h2_compagnies["taux_retard"] *= 100
h2_compagnies = h2_compagnies.sort_values("part_vols_longs", ascending=False).reset_index(drop=True)

h2_compagnies[["AIRLINE", "compagnie", "vols", "part_vols_longs",
               "taux_retard", "retard_moyen", "distance_moyenne"]].round(2)

,AIRLINE,compagnie,vols,part_vols_longs,taux_retard,retard_moyen,distance_moyenne
0,VX,Virgin America,61248,43.20,19.23,4.74,1404.38
1,UA,United Air Lines Inc.,507762,33.92,20.62,5.43,1271.68
2,AS,Alaska Airlines Inc.,171439,27.76,13.04,-0.98,1198.69
3,B6,JetBlue Airways,262042,20.84,22.58,6.68,1063.05
4,HA,Hawaiian Airlines Inc.,76041,19.59,11.33,2.02,632.03
5,US,US Airways Inc.,194223,19.52,18.82,3.71,915.38
6,AA,American Airlines Inc.,712935,19.33,18.27,3.45,1042.37
7,DL,Delta Air Lines Inc.,870275,16.98,13.56,0.19,853.60
8,NK,Spirit Air Lines,115193,14.37,29.71,14.47,985.78
9,F9,Frontier Airlines Inc.,90090,13.85,26.16,12.50,967.16


### 7.1 Y a-t-il une relation entre part de vols longs et retard ?

Test sur les 14 compagnies : correlation de Pearson (relation lineaire), de Spearman (relation monotone,
robuste aux valeurs extremes) et regression lineaire pour chiffrer la pente.

In [22]:
x = h2_compagnies.part_vols_longs.to_numpy()
y = h2_compagnies.taux_retard.to_numpy()

r_p, p_p = stats.pearsonr(x, y)
r_s, p_s = stats.spearmanr(x, y)
reg = stats.linregress(x, y)

H2_CORR = {
    "n_compagnies": len(h2_compagnies),
    "pearson_r": round(float(r_p), 3), "pearson_p": round(float(p_p), 3),
    "spearman_rho": round(float(r_s), 3), "spearman_p": round(float(p_s), 3),
    "pente": round(float(reg.slope), 4), "r2": round(float(reg.rvalue ** 2), 3),
    "regression_p": round(float(reg.pvalue), 3),
}

print(f"n = {H2_CORR['n_compagnies']} compagnies")
print(f"Pearson  r   = {H2_CORR['pearson_r']:+.3f}  (p = {H2_CORR['pearson_p']:.3f})")
print(f"Spearman rho = {H2_CORR['spearman_rho']:+.3f}  (p = {H2_CORR['spearman_p']:.3f})")
print(f"Regression   : {H2_CORR['pente']:+.4f} point de retard par point de vols longs, "
      f"R2 = {H2_CORR['r2']:.3f}, p = {H2_CORR['regression_p']:.3f}")
print()
print(f"La part de vols longs explique {H2_CORR['r2'] * 100:.1f} % de la variance du taux de retard "
      f"entre compagnies. Le signe est {'negatif' if r_p < 0 else 'positif'}, "
      f"soit {'l inverse de' if r_p < 0 else 'le sens de'} l'hypothese.")

n = 14 compagnies
Pearson  r   = -0.179  (p = 0.539)
Spearman rho = -0.222  (p = 0.445)
Regression   : -0.0700 point de retard par point de vols longs, R2 = 0.032, p = 0.539

La part de vols longs explique 3.2 % de la variance du taux de retard entre compagnies. Le signe est negatif, soit l inverse de l'hypothese.


### 7.2 Contre-epreuve au niveau du vol

La correlation entre compagnies peut etre brouillee par tout ce qui les distingue par ailleurs. On regarde
donc directement, vol par vol, ce que la distance fait au retard.

In [23]:
BORNES = [0, 500, 1000, 1500, 2000, 10_000]
ETIQUETTES = ["< 500 mi", "500 - 999 mi", "1000 - 1499 mi", "1500 - 1999 mi", ">= 2000 mi"]
tranche = pd.cut(exploitables.DISTANCE, bins=BORNES, labels=ETIQUETTES, right=False)

h2_distance = exploitables.groupby(tranche, observed=True).agg(
    vols=("DISTANCE", "size"),
    taux_retard=("EST_EN_RETARD", "mean"),
    retard_moyen=("ARRIVAL_DELAY", "mean"),
    retard_depart_moyen=("DEPARTURE_DELAY", "mean"),
).reset_index()
h2_distance.columns = ["tranche", "vols", "taux_retard", "retard_moyen", "retard_depart_moyen"]
h2_distance["taux_retard"] *= 100
h2_distance["rattrapage_min"] = h2_distance.retard_depart_moyen - h2_distance.retard_moyen
h2_distance.round(2).to_string(index=False)

'       tranche    vols  taux_retard  retard_moyen  retard_depart_moyen  rattrapage_min\n      < 500 mi 2073950        18.21          5.16                 8.05            2.90\n  500 - 999 mi 2015991        18.65          4.64                 9.53            4.89\n1000 - 1499 mi  845177        19.72          4.49                10.89            6.40\n1500 - 1999 mi  409263        18.34          2.29                10.50            8.21\n    >= 2000 mi  369627        18.42          1.07                10.00            8.93'

In [24]:
court = h2_distance.iloc[0]
long_ = h2_distance.iloc[-1]
print(f"Taux de retard  : {court.tranche} = {court.taux_retard:.2f} %  vs  "
      f"{long_.tranche} = {long_.taux_retard:.2f} %  ({long_.taux_retard - court.taux_retard:+.2f} pt)")
print(f"Retard moyen    : {court.retard_moyen:.2f} min  vs  {long_.retard_moyen:.2f} min "
      f"({long_.retard_moyen - court.retard_moyen:+.2f} min)")
print(f"Minutes rattrapees en vol : {court.rattrapage_min:.2f} min  vs  {long_.rattrapage_min:.2f} min")

Taux de retard  : < 500 mi = 18.21 %  vs  >= 2000 mi = 18.42 %  (+0.22 pt)
Retard moyen    : 5.16 min  vs  1.07 min (-4.09 min)
Minutes rattrapees en vol : 2.90 min  vs  8.93 min


**Le mecanisme apparait ici.** Le taux de retard est quasi plat d'une tranche de distance a l'autre, mais le
retard **moyen** chute fortement quand la distance augmente. La difference entre retard au depart et retard
a l'arrivee montre pourquoi : plus un vol est long, plus il **rattrape** de minutes en vol, parce que les
horaires publies integrent une marge proportionnelle au temps de vol.

Autrement dit, la distance n'aggrave pas le retard a l'arrivee : elle donne au contraire de la marge pour
l'absorber. L'hypothese H2 partait d'une intuition raisonnable, que les donnees ne valident pas.

### 7.3 Comparaison des deux groupes de compagnies

On coupe malgre tout les 14 compagnies a la mediane de part de vols longs, pour montrer explicitement ce
que produit — et ce que vaut — la comparaison qu'on aurait pu faire d'emblee.

In [25]:
MEDIANE_PART = float(h2_compagnies.part_vols_longs.median())
groupe_fort = set(h2_compagnies.loc[h2_compagnies.part_vols_longs >= MEDIANE_PART, "AIRLINE"])
vols["GROUPE_LONG"] = vols.AIRLINE.isin(groupe_fort)

h2_groupes = exploitables.groupby(vols.GROUPE_LONG, observed=True).agg(
    vols=("EST_EN_RETARD", "size"), taux_retard=("EST_EN_RETARD", "mean")).reset_index()
h2_groupes["segment"] = np.where(h2_groupes.GROUPE_LONG,
                                 "Part forte de vols longs", "Part faible de vols longs")
h2_groupes["taux_retard"] *= 100

TAUX_FORT = float(h2_groupes.loc[h2_groupes.GROUPE_LONG, "taux_retard"].iloc[0])
TAUX_FAIBLE = float(h2_groupes.loc[~h2_groupes.GROUPE_LONG, "taux_retard"].iloc[0])
H2_TEST = test_chi2(vols.GROUPE_LONG, vols.EST_EN_RETARD, exploitables)

print(f"Mediane de part de vols longs : {MEDIANE_PART:.2f} %")
print(f"Part forte  : {TAUX_FORT:.2f} % de retards")
print(f"Part faible : {TAUX_FAIBLE:.2f} % de retards")
print(f"Ecart       : {TAUX_FORT - TAUX_FAIBLE:+.2f} point")
print()
print(f"chi2 = {H2_TEST['chi2']:,.1f}  p = {H2_TEST['p_value']:.3g}  V de Cramer = {H2_TEST['cramer_v']}")
print(f"-> p tres significative pour un ecart de {TAUX_FORT - TAUX_FAIBLE:.2f} point et un V de "
      f"{H2_TEST['cramer_v']} : c'est exactement l'artefact de taille d'echantillon annonce plus haut.")
print(f"   Le test valide au niveau compagnie (n = 14) donne, lui, p = {H2_CORR['spearman_p']:.3f}.")

Mediane de part de vols longs : 18.16 %
Part forte  : 18.81 % de retards
Part faible : 18.51 % de retards
Ecart       : +0.30 point

chi2 = 78.1  p = 1e-18  V de Cramer = 0.0037
-> p tres significative pour un ecart de 0.30 point et un V de 0.0037 : c'est exactement l'artefact de taille d'echantillon annonce plus haut.
   Le test valide au niveau compagnie (n = 14) donne, lui, p = 0.445.


### 7.4 Conclusion H2

In [26]:
H2_STATUT = "Non validee"
H2_RESUME = (
    f"Sur les {H2_CORR['n_compagnies']} compagnies, la correlation entre part de vols longs et taux de retard "
    f"est de {H2_CORR['pearson_r']:+.3f} (Pearson, p = {H2_CORR['pearson_p']:.2f}) et "
    f"{H2_CORR['spearman_rho']:+.3f} (Spearman, p = {H2_CORR['spearman_p']:.2f}) : elle est faible, de signe "
    f"contraire a l'hypothese, et non significative. La part de vols longs n'explique que "
    f"{H2_CORR['r2'] * 100:.1f} % des ecarts de ponctualite entre compagnies. La comparaison des deux groupes "
    f"donne {TAUX_FORT:.2f} % contre {TAUX_FAIBLE:.2f} %, soit {TAUX_FORT - TAUX_FAIBLE:+.2f} point, "
    f"un ecart sans portee pratique. Au niveau du vol, la distance ne degrade pas la ponctualite : les vols "
    f"les plus longs rattrapent en moyenne {float(h2_distance.iloc[-1].rattrapage_min):.1f} minutes en vol "
    f"contre {float(h2_distance.iloc[0].rattrapage_min):.1f} pour les plus courts."
)
print(H2_STATUT.upper()); print(); print(H2_RESUME)

NON VALIDEE

Sur les 14 compagnies, la correlation entre part de vols longs et taux de retard est de -0.179 (Pearson, p = 0.54) et -0.222 (Spearman, p = 0.45) : elle est faible, de signe contraire a l'hypothese, et non significative. La part de vols longs n'explique que 3.2 % des ecarts de ponctualite entre compagnies. La comparaison des deux groupes donne 18.81 % contre 18.51 %, soit +0.30 point, un ecart sans portee pratique. Au niveau du vol, la distance ne degrade pas la ponctualite : les vols les plus longs rattrapent en moyenne 8.9 minutes en vol contre 2.9 pour les plus courts.


**Reponse a l'hypothese : non validee.**

Ce n'est pas un echec de l'analyse, c'est un resultat : la structure de reseau d'une compagnie n'explique
pas sa ponctualite. Ce qui la separe se voit ailleurs dans le tableau des 14 compagnies — Spirit et Frontier,
compagnies low-cost a rotation tres tendue, sont les plus en retard malgre une part de vols longs faible ;
Delta et Alaska sont les plus ponctuelles avec des profils de reseau opposes. La variable explicative est le
**modele d'exploitation**, pas la distance.

**Ce que ca implique pour le dashboard** : deux barres a 18,8 % et 18,5 % donneraient a voir une difference
qui n'existe pas. Il faut un nuage de points sur les 14 compagnies, ou l'absence de relation se lit
directement, avec les compagnies nommees.

---
# 8. H3 — « Une petite minorite de liaisons concentre la majorite des annulations »

Une route est un couple oriente `ORIGINE -> DESTINATION`. Perimetre : les onze mois en codes IATA
(cf. section 3.2).

Le piege de cette hypothese est qu'elle est **presque vraie par construction** : le trafic aerien est lui-meme
tres concentre, donc les routes qui portent le plus d'annulations sont d'abord celles qui portent le plus de
vols. Pour que le resultat ait un sens, il faut comparer la concentration des annulations a **la concentration
du trafic**, et regarder les **taux** et pas seulement les volumes.

In [27]:
base_routes = vols[vols.ROUTE_LISIBLE]

routes = base_routes.groupby("ROUTE", observed=True).agg(
    vols=("CANCELLED", "size"),
    annulations=("CANCELLED", "sum"),
    distance=("DISTANCE", "mean"),
).reset_index()
routes["taux_annulation"] = routes.annulations / routes.vols * 100
routes[["origine", "destination"]] = routes.ROUTE.str.split(" -> ", expand=True)
routes = routes.sort_values(["annulations", "vols"], ascending=False).reset_index(drop=True)
routes["rang"] = np.arange(1, len(routes) + 1)
routes["part_routes_cumulee"] = routes.rang / len(routes) * 100
routes["part_annulations_cumulee"] = routes.annulations.cumsum() / routes.annulations.sum() * 100

N_ROUTES = len(routes)
N_ANNUL = int(routes.annulations.sum())
print(f"Routes distinctes  : {N_ROUTES:,}")
print(f"Annulations        : {N_ANNUL:,}  ({N_ANNUL / int(vols.CANCELLED.sum()) * 100:.1f} % du total annuel)")
print(f"Vols du perimetre  : {int(routes.vols.sum()):,}")
routes.head(10)[["ROUTE", "vols", "annulations", "taux_annulation", "part_annulations_cumulee"]].round(2)

Routes distinctes  : 4,693
Annulations        : 87,430  (97.3 % du total annuel)
Vols du perimetre  : 5,332,914


,ROUTE,vols,annulations,taux_annulation,part_annulations_cumulee
0,BOS -> LGA,7096,443,6.24,0.51
1,LGA -> BOS,7100,441,6.21,1.01
2,LGA -> ORD,9639,435,4.51,1.51
3,ORD -> LGA,9575,408,4.26,1.98
4,LAX -> SFO,13457,342,2.54,2.37
5,LGA -> DCA,4303,340,7.90,2.76
6,SFO -> LAX,13744,338,2.46,3.14
7,DCA -> LGA,4303,334,7.76,3.52
8,DCA -> BOS,7686,271,3.53,3.83
9,BOS -> DCA,7687,266,3.46,4.14


### 8.1 Concentration des annulations, comparee a celle du trafic

In [28]:
def courbe_lorenz(valeurs):
    """Part cumulee de la grandeur, triee du plus gros au plus petit."""
    v = np.sort(np.asarray(valeurs, dtype=float))[::-1]
    return np.cumsum(v) / v.sum() * 100

def gini(valeurs):
    """Indice de Gini : 0 = tout le monde pareil, 1 = tout concentre sur un seul."""
    v = np.sort(np.asarray(valeurs, dtype=float))
    n = len(v)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(v) / (n * v.sum()))

cum_annulations = courbe_lorenz(routes.annulations)
cum_vols = courbe_lorenz(routes.vols)
part_routes = np.arange(1, N_ROUTES + 1) / N_ROUTES * 100

idx_50 = int(np.argmax(cum_annulations >= 50))
N_ROUTES_50 = idx_50 + 1
PCT_ROUTES_50 = N_ROUTES_50 / N_ROUTES * 100

H3_CONC = {
    "routes": N_ROUTES, "annulations": N_ANNUL,
    "routes_pour_50pct": N_ROUTES_50, "pct_routes_pour_50pct": round(PCT_ROUTES_50, 2),
    "gini_annulations": round(gini(routes.annulations), 3),
    "gini_vols": round(gini(routes.vols), 3),
}
for seuil in (10, 20, 30):
    k = int(np.ceil(N_ROUTES * seuil / 100)) - 1
    H3_CONC[f"annulations_top{seuil}pct"] = round(float(cum_annulations[k]), 2)
    H3_CONC[f"vols_top{seuil}pct"] = round(float(cum_vols[k]), 2)

print(f"{N_ROUTES_50:,} routes ({PCT_ROUTES_50:.2f} %) portent 50 % des annulations.")
print()
print(f"{'seuil':>8} | {'% annulations':>14} | {'% des vols':>11} | {'ecart':>7}")
print("-" * 50)
for seuil in (10, 20, 30):
    a, v = H3_CONC[f"annulations_top{seuil}pct"], H3_CONC[f"vols_top{seuil}pct"]
    print(f"top {seuil:>2} % | {a:>13.2f} % | {v:>10.2f} % | {a - v:>+6.2f}")
print()
print(f"Gini des annulations : {H3_CONC['gini_annulations']:.3f}")
print(f"Gini du trafic       : {H3_CONC['gini_vols']:.3f}")

455 routes (9.70 %) portent 50 % des annulations.

   seuil |  % annulations |  % des vols |   ecart
--------------------------------------------------
top 10 % |         50.89 % |      38.72 % | +12.17
top 20 % |         70.42 % |      58.08 % | +12.34
top 30 % |         81.94 % |      71.13 % | +10.81

Gini des annulations : 0.683
Gini du trafic       : 0.558


**Lecture indispensable.** Enonce seul, « 10 % des routes portent 51 % des annulations » impressionne. Mais
ces memes 10 % de routes portent deja **39 % des vols**. L'essentiel de la concentration est donc mecanique :
il reflete la concentration du trafic sur les grands axes, pas une fragilite propre a ces routes.

L'information reelle est dans l'**ecart entre les deux courbes**, confirme par les Gini (0,68 pour les
annulations contre 0,56 pour le trafic). Il existe bien un sur-risque, mais il est plus modeste que le chiffre
brut ne le laisse croire. Pour le mesurer proprement, on passe aux taux.

### 8.2 Le vrai signal : le taux d'annulation, pas le volume

In [29]:
prioritaires = routes.iloc[:N_ROUTES_50]
autres = routes.iloc[N_ROUTES_50:]

TAUX_PRIO = prioritaires.annulations.sum() / prioritaires.vols.sum() * 100
TAUX_AUTRES = autres.annulations.sum() / autres.vols.sum() * 100
SURRISQUE = TAUX_PRIO / TAUX_AUTRES

H3_CONC.update({
    "taux_annulation_prioritaires": round(float(TAUX_PRIO), 2),
    "taux_annulation_autres": round(float(TAUX_AUTRES), 2),
    "surrisque": round(float(SURRISQUE), 2),
})

print(f"{N_ROUTES_50} routes prioritaires : {TAUX_PRIO:.2f} % d'annulation "
      f"({int(prioritaires.annulations.sum()):,} annulations sur {int(prioritaires.vols.sum()):,} vols)")
print(f"{len(autres):,} autres routes      : {TAUX_AUTRES:.2f} % d'annulation")
print(f"\nSur-risque : x{SURRISQUE:.2f}")
print("\nC'est ce chiffre qui resiste a l'objection du volume : a nombre de vols egal, une route")
print("prioritaire est annulee deux a trois fois plus souvent qu'une autre.")

455 routes prioritaires : 2.92 % d'annulation (43,736 annulations sur 1,500,205 vols)
4,238 autres routes      : 1.14 % d'annulation

Sur-risque : x2.56

C'est ce chiffre qui resiste a l'objection du volume : a nombre de vols egal, une route
prioritaire est annulee deux a trois fois plus souvent qu'une autre.


In [30]:
# Quand ces routes sont-elles annulees ? Et sont-elles concentrees geographiquement ?
prio_set = set(prioritaires.ROUTE)
detail_prio = base_routes[base_routes.ROUTE.isin(prio_set)]

saison = detail_prio.groupby("MONTH", observed=True).CANCELLED.agg(["size", "sum"])
saison["taux"] = saison["sum"] / saison["size"] * 100
PART_HIVER = float(saison.loc[saison.index.isin([1, 2, 3]), "sum"].sum() / saison["sum"].sum() * 100)

aeroports_annul = base_routes.groupby(base_routes.ROUTE.str[:3], observed=True).agg(
    vols=("CANCELLED", "size"), annulations=("CANCELLED", "sum")).reset_index()
aeroports_annul.columns = ["aeroport", "vols", "annulations"]
aeroports_annul["taux_annulation"] = aeroports_annul.annulations / aeroports_annul.vols * 100
top_aeroports = aeroports_annul.nlargest(10, "annulations").reset_index(drop=True)

print(f"Part des annulations des routes prioritaires survenue en janvier-mars : {PART_HIVER:.1f} %")
print(f"(pour reference, janvier-mars = {base_routes.MONTH.isin([1, 2, 3]).mean() * 100:.1f} % des vols)")
print()
print("Aeroports d'origine les plus touches :")
print(top_aeroports.round(2).to_string(index=False))

Part des annulations des routes prioritaires survenue en janvier-mars : 51.7 %
(pour reference, janvier-mars = 26.3 % des vols)

Aeroports d'origine les plus touches :
aeroport   vols  annulations  taux_annulation
     ORD 285884         8548             2.99
     DFW 239551         6254             2.61
     LGA  99605         4531             4.55
     EWR 101772         3110             3.06
     BOS 107847         2654             2.46
     ATL 346836         2557             0.74
     LAX 194673         2164             1.11
     SFO 148008         2148             1.45
     IAH 146622         2130             1.45
     DEN 196055         2123             1.08


**Le phenomene a un nom.** Plus de la moitie des annulations des routes prioritaires tombent sur le premier
trimestre, alors que ce trimestre ne represente qu'un quart des vols. Et les aeroports en tete ne sont pas les
plus gros du pays — Atlanta, premier aeroport mondial en trafic, affiche un taux de 0,74 %, quand LaGuardia
est a 4,55 %.

H3 ne decrit donc pas « des routes fragiles » en general : elle decrit la **navette court-courrier du Nord-Est
en hiver** (New York, Boston, Washington, Chicago), ou des rotations tres frequentes sur de courtes distances
rencontrent la saison des tempetes. C'est coherent avec le fait que la meteo est le premier motif d'annulation
du dataset, et avec les pires journees identifiees en section 5.

In [31]:
# Table des routes a afficher, enrichie des coordonnees pour la carte du dashboard.
coords = (aeroports_ref.dropna(subset=["LATITUDE", "LONGITUDE"])
          .set_index("IATA_CODE")[["AIRPORT", "CITY", "STATE", "LATITUDE", "LONGITUDE"]])

routes_geo = routes.join(coords.add_prefix("o_"), on="origine").join(coords.add_prefix("d_"), on="destination")
sans_coords = routes_geo.o_LATITUDE.isna() | routes_geo.d_LATITUDE.isna()
print(f"Routes sans coordonnees completes : {int(sans_coords.sum())} sur {N_ROUTES} "
      f"({routes_geo.loc[sans_coords, 'annulations'].sum() / N_ANNUL * 100:.2f} % des annulations)")

TOP_ROUTES = routes.head(15)[["rang", "ROUTE", "origine", "destination", "vols",
                              "annulations", "taux_annulation", "distance"]].copy()
TOP_ROUTES["ville_origine"] = TOP_ROUTES.origine.map(coords.CITY)
TOP_ROUTES["ville_destination"] = TOP_ROUTES.destination.map(coords.CITY)
TOP_ROUTES.round(2)

Routes sans coordonnees completes : 22 sur 4693 (0.07 % des annulations)


,rang,ROUTE,origine,destination,vols,annulations,taux_annulation,distance,ville_origine,ville_destination
0,1,BOS -> LGA,BOS,LGA,7096,443,6.24,184.0,Boston,New York
1,2,LGA -> BOS,LGA,BOS,7100,441,6.21,184.0,New York,Boston
2,3,LGA -> ORD,LGA,ORD,9639,435,4.51,733.0,New York,Chicago
3,4,ORD -> LGA,ORD,LGA,9575,408,4.26,733.0,Chicago,New York
4,5,LAX -> SFO,LAX,SFO,13457,342,2.54,337.0,Los Angeles,San Francisco
5,6,LGA -> DCA,LGA,DCA,4303,340,7.90,214.0,New York,Arlington
6,7,SFO -> LAX,SFO,LAX,13744,338,2.46,337.0,San Francisco,Los Angeles
7,8,DCA -> LGA,DCA,LGA,4303,334,7.76,214.0,Arlington,New York
8,9,DCA -> BOS,DCA,BOS,7686,271,3.53,399.0,Arlington,Boston
9,10,BOS -> DCA,BOS,DCA,7687,266,3.46,399.0,Boston,Arlington


In [32]:
# Routes au taux d'annulation le plus eleve : on impose un volume minimum, sinon une route
# a 4 vols dont 1 annule sortirait en tete avec 25 %.
VOLUME_MINIMUM = 365     # environ un vol par jour sur l'annee
routes_risque = (routes[routes.vols >= VOLUME_MINIMUM]
                 .nlargest(12, "taux_annulation")
                 .reset_index(drop=True))
print(f"Routes >= {VOLUME_MINIMUM} vols retenues : {int((routes.vols >= VOLUME_MINIMUM).sum()):,} sur {N_ROUTES:,}")
routes_risque[["ROUTE", "vols", "annulations", "taux_annulation"]].round(2)

Routes >= 365 vols retenues : 3,136 sur 4,693


,ROUTE,vols,annulations,taux_annulation
0,ORF -> LGA,402,61,15.17
1,LGA -> ORF,515,68,13.20
2,DCA -> JFK,996,114,11.45
3,JFK -> DCA,1002,106,10.58
4,RDU -> LGA,1878,197,10.49
5,LGA -> BHM,490,51,10.41
6,CHO -> LGA,396,41,10.35
7,LGA -> GSO,1295,134,10.35
8,LGA -> DAY,403,41,10.17
9,GSO -> LGA,1295,129,9.96


### 8.3 Robustesse du resultat

In [33]:
# 1) Le resultat depend-il du perimetre IATA (= de l'exclusion d'octobre) ?
routes_hors_oct = (base_routes[base_routes.MONTH.ne(10)]
                   .groupby("ROUTE", observed=True).CANCELLED.agg(["size", "sum"]))
c = np.sort(routes_hors_oct["sum"].to_numpy())[::-1]
cum = np.cumsum(c) / c.sum() * 100
pct_sans_oct = (int(np.argmax(cum >= 50)) + 1) / len(c) * 100

# 2) Le resultat depend-il du choix de compter les routes orientees (A->B et B->A separement) ?
paire = base_routes.assign(
    PAIRE=[" <-> ".join(sorted(p)) for p in zip(base_routes.ROUTE.str[:3], base_routes.ROUTE.str[-3:])])
g_paire = paire.groupby("PAIRE", observed=True).CANCELLED.sum()
c2 = np.sort(g_paire.to_numpy())[::-1]
cum2 = np.cumsum(c2) / c2.sum() * 100
pct_paires = (int(np.argmax(cum2 >= 50)) + 1) / len(c2) * 100

print(f"Reference (routes orientees, 11 mois)   : 50 % des annulations dans {PCT_ROUTES_50:.2f} % des routes")
print(f"Variante  (octobre exclu explicitement) : {pct_sans_oct:.2f} %  -> identique par construction,")
print(f"                                          le filtre IATA equivaut exactement a retirer octobre")
print(f"Variante  (liaisons non orientees)      : {pct_paires:.2f} % des {len(c2):,} liaisons")
print(f"\nLe resultat ne depend pas de ces choix de perimetre.")

Reference (routes orientees, 11 mois)   : 50 % des annulations dans 9.70 % des routes
Variante  (octobre exclu explicitement) : 9.70 %  -> identique par construction,
                                          le filtre IATA equivaut exactement a retirer octobre
Variante  (liaisons non orientees)      : 9.62 % des 2,381 liaisons

Le resultat ne depend pas de ces choix de perimetre.


### 8.4 Conclusion H3

In [34]:
H3_STATUT = "Confirmee, avec une reserve importante"
H3_RESUME = (
    f"{nb_fr(N_ROUTES_50)} routes sur {nb_fr(N_ROUTES)} ({PCT_ROUTES_50:.2f} %) portent la moitie des "
    f"{nb_fr(N_ANNUL)} annulations du perimetre, et le Gini des annulations ({H3_CONC['gini_annulations']:.3f}) "
    f"depasse celui du trafic ({H3_CONC['gini_vols']:.3f}). La concentration est donc reelle, mais elle est "
    f"pour une bonne part mecanique : les 10 % de routes en tete portent "
    f"{H3_CONC['annulations_top10pct']:.1f} % des annulations et deja {H3_CONC['vols_top10pct']:.1f} % des vols. "
    f"Le resultat solide est le sur-risque a volume comparable : {TAUX_PRIO:.2f} % d'annulation sur les routes "
    f"prioritaires contre {TAUX_AUTRES:.2f} % ailleurs, soit x{SURRISQUE:.2f}. Ces routes sont majoritairement "
    f"des navettes courtes du Nord-Est, et {PART_HIVER:.0f} % de leurs annulations tombent au premier trimestre."
)
print(H3_STATUT.upper()); print(); print(H3_RESUME)

CONFIRMEE, AVEC UNE RESERVE IMPORTANTE

455 routes sur 4 693 (9.70 %) portent la moitie des 87 430 annulations du perimetre, et le Gini des annulations (0.683) depasse celui du trafic (0.558). La concentration est donc reelle, mais elle est pour une bonne part mecanique : les 10 % de routes en tete portent 50.9 % des annulations et deja 38.7 % des vols. Le resultat solide est le sur-risque a volume comparable : 2.92 % d'annulation sur les routes prioritaires contre 1.14 % ailleurs, soit x2.56. Ces routes sont majoritairement des navettes courtes du Nord-Est, et 52 % de leurs annulations tombent au premier trimestre.


**Reponse a l'hypothese : confirmee, mais le chiffre spectaculaire doit etre relativise.**

Oui, une minorite de routes concentre la majorite des annulations. Mais presenter « 9,7 % des routes = 50 %
des annulations » sans preciser que ces routes portent deja une part enorme du trafic serait un raccourci
trompeur. Le resultat defendable est le **sur-risque a volume comparable**, et son explication : un effet
conjoint de reseau court-courrier dense et de meteo hivernale sur le Nord-Est.

**Ce que ca implique pour le dashboard** : la carte seule ne demontre rien. Il faut afficher les deux courbes
de concentration superposees — l'ecart entre elles *est* la demonstration.

---
## 9. Figures

Les memes graphiques que le dashboard, en PNG, pour le rapport ecrit et le support de presentation.
Ils sont enregistres dans `v2/outputs/figures/`.

In [35]:
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 140, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9d3d7", "axes.labelcolor": "#4a5c63",
    "text.color": "#172126", "xtick.color": "#4a5c63", "ytick.color": "#4a5c63",
    "grid.color": "#e3e9eb",
})
TEAL, ORANGE, GRIS, BLEU, JAUNE = "#087f8c", "#e45c3a", "#94a3b8", "#3977a8", "#d69e2e"

def enregistrer(fig, nom):
    chemin = FIG / f"{nom}.png"
    fig.savefig(chemin)
    plt.close(fig)
    print("figure ->", chemin.relative_to(RACINE))

In [36]:
# Figure 1 - H1 : le rythme des retards jour par jour
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(journalier.DATE, journalier.taux_retard, color=TEAL, lw=1.1)
for nom, (masque, _) in FENETRES.items():
    jours = vols.loc[masque, "DATE"]
    ax.axvspan(jours.min(), jours.max(), color=ORANGE, alpha=0.10, lw=0)
ax.axhline(TAUX_RETARD_GLOBAL, color=GRIS, ls="--", lw=1)
ax.annotate(f"moyenne annuelle {TAUX_RETARD_GLOBAL:.1f} %",
            (journalier.DATE.iloc[262], TAUX_RETARD_GLOBAL), textcoords="offset points",
            xytext=(0, -14), color="#4a5c63", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=.85))
for _, r in PIRES_RETARDS.head(3).iterrows():
    ax.annotate(r.DATE.strftime("%d %b"), (r.DATE, r.taux_retard), textcoords="offset points",
                xytext=(0, 7), ha="center", fontsize=8, color=ORANGE, fontweight="bold")
ax.set_title("H1 - Taux de vols en retard, jour par jour (zones orangees : fenetres de forte mobilite)")
ax.set_ylabel("% de vols en retard"); ax.grid(axis="y", alpha=.6)
enregistrer(fig, "h1_rythme_journalier")

figure -> v2\outputs\figures\h1_rythme_journalier.png


In [37]:
# Figure 2 - H1 : effet net de chaque fenetre, une fois la reference neutralisee
fig, ax = plt.subplots(figsize=(7.5, 3.6))
d_ = h1_controle.sort_values("effet_net_points")
couleurs = [ORANGE if v > 0.5 else (BLEU if v < -0.5 else GRIS) for v in d_.effet_net_points]
barres = ax.barh(d_.fenetre, d_.effet_net_points, color=couleurs, height=.62)
ax.axvline(0, color="#4a5c63", lw=1.2)
for b, v in zip(barres, d_.effet_net_points):
    ax.text(v + (0.5 if v > 0 else -0.5), b.get_y() + b.get_height() / 2, f"{v:+.1f} pt",
            va="center", ha="left" if v > 0 else "right", fontsize=9, fontweight="bold")
ax.set_xlim(min(d_.effet_net_points) - 5, max(d_.effet_net_points) + 5)
ax.set_title("H1 - Effet net sur le taux de retard, vs periode de reference comparable")
ax.set_xlabel("ecart en points de %"); ax.grid(axis="x", alpha=.6)
enregistrer(fig, "h1_effet_net_fenetres")

figure -> v2\outputs\figures\h1_effet_net_fenetres.png


In [38]:
# Figure 3 - H2 : absence de relation entre part de vols longs et retard
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.scatter(h2_compagnies.part_vols_longs, h2_compagnies.taux_retard,
           s=h2_compagnies.vols / 4000, color=TEAL, alpha=.75, edgecolor="white", zorder=3)
xs = np.linspace(0, h2_compagnies.part_vols_longs.max() * 1.08, 50)
ax.plot(xs, reg.intercept + reg.slope * xs, color=ORANGE, ls="--", lw=1.6, zorder=2)
for _, r in h2_compagnies.iterrows():
    ax.annotate(r.AIRLINE, (r.part_vols_longs, r.taux_retard), textcoords="offset points",
                xytext=(7, 4), fontsize=8.5, fontweight="bold", color="#172126")
ax.text(.98, .96, f"r = {H2_CORR['pearson_r']:+.3f}   p = {H2_CORR['pearson_p']:.2f}\n"
                  f"R2 = {H2_CORR['r2']:.3f}   n = {H2_CORR['n_compagnies']}",
        transform=ax.transAxes, ha="right", va="top", fontsize=9, color="#4a5c63",
        bbox=dict(boxstyle="round,pad=0.45", fc="#f4f7f8", ec="#dbe3e6"))
ax.set_title("H2 - Aucune relation entre part de vols longs et ponctualite")
ax.set_xlabel("part de vols >= 1500 miles (%)"); ax.set_ylabel("% de vols en retard")
ax.grid(alpha=.6)
enregistrer(fig, "h2_nuage_compagnies")

figure -> v2\outputs\figures\h2_nuage_compagnies.png


In [39]:
# Figure 4 - H2 : ce que fait vraiment la distance
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.bar(h2_distance.tranche.astype(str), h2_distance.taux_retard, color=TEAL, width=.62)
a1.axhline(TAUX_RETARD_GLOBAL, color=GRIS, ls="--", lw=1)
a1.set_title("Taux de retard : quasi plat"); a1.set_ylabel("% de vols en retard")
a1.set_ylim(0, 26); a1.grid(axis="y", alpha=.6)
a2.bar(h2_distance.tranche.astype(str), h2_distance.rattrapage_min, color=ORANGE, width=.62)
a2.set_title("Minutes rattrapees en vol : croissantes"); a2.set_ylabel("minutes")
a2.grid(axis="y", alpha=.6)
for a in (a1, a2):
    a.tick_params(axis="x", rotation=20, labelsize=8.5)
fig.suptitle("H2 - La distance ne degrade pas l'arrivee, elle donne de la marge pour absorber le retard",
             fontsize=11, fontweight="bold", y=1.04)
enregistrer(fig, "h2_effet_distance")

figure -> v2\outputs\figures\h2_effet_distance.png


In [40]:
# Figure 5 - H3 : concentration des annulations vs concentration du trafic
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.fill_between(part_routes, cum_vols, cum_annulations, color=ORANGE, alpha=.13,
                label="sur-concentration des annulations")
ax.plot(part_routes, cum_annulations, color=ORANGE, lw=2.4, label="annulations cumulees")
ax.plot(part_routes, cum_vols, color=BLEU, lw=2.0, ls="-", label="vols cumules")
ax.plot([0, 100], [0, 100], color=GRIS, ls=":", lw=1.2, label="repartition uniforme")
ax.axhline(50, color=GRIS, ls="--", lw=.9)
ax.axvline(PCT_ROUTES_50, color=GRIS, ls="--", lw=.9)
ax.annotate(f"{PCT_ROUTES_50:.1f} % des routes\n= 50 % des annulations",
            (PCT_ROUTES_50, 50), textcoords="offset points", xytext=(14, -34), fontsize=9,
            color="#172126", arrowprops=dict(arrowstyle="->", color="#4a5c63", lw=.9))
ax.set_title("H3 - La concentration des annulations depasse celle du trafic, mais de peu")
ax.set_xlabel("part cumulee des routes (%)"); ax.set_ylabel("part cumulee (%)")
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.grid(alpha=.6); ax.legend(fontsize=8.5, loc="lower right")
enregistrer(fig, "h3_concentration_lorenz")

figure -> v2\outputs\figures\h3_concentration_lorenz.png


In [41]:
# Figure 6 - H3 : ou et quand
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.0))
t = TOP_ROUTES.head(10).iloc[::-1]
a1.barh(t.ROUTE, t.annulations, color=ORANGE, height=.66)
for i, (n, tx) in enumerate(zip(t.annulations, t.taux_annulation)):
    a1.text(n + 6, i, f"{tx:.1f} %", va="center", fontsize=8.5, color="#4a5c63")
a1.set_title("Routes les plus annulees"); a1.set_xlabel("vols annules"); a1.grid(axis="x", alpha=.6)
a2.bar(saison.index, saison["taux"], color=[ORANGE if m in (1, 2, 3) else TEAL for m in saison.index], width=.66)
a2.set_title(f"Taux d'annulation des {N_ROUTES_50} routes prioritaires, par mois")
a2.set_xlabel("mois"); a2.set_ylabel("% annules"); a2.set_xticks(range(1, 13)); a2.grid(axis="y", alpha=.6)
fig.suptitle("H3 - Une navette court-courrier du Nord-Est, concentree sur l'hiver",
             fontsize=11, fontweight="bold", y=1.02)
enregistrer(fig, "h3_routes_et_saison")

figure -> v2\outputs\figures\h3_routes_et_saison.png


---
## 10. Exports

Trois familles de livrables :

1. **CSV** — tableaux reutilisables dans Excel, Tableau ou le rapport ecrit.
2. **`resume_v2.json`** — synthese chiffree de toutes les hypotheses.
3. **`dashboard/dashboard_data.js`** — les donnees du dashboard HTML.

Sur ce troisieme point : le dashboard ne contient **aucun chiffre en dur**. Il lit ce fichier, qui est
regenere ici. Reexecuter le notebook met donc le dashboard a jour automatiquement, et il est impossible
qu'il affiche autre chose que ce que le notebook a calcule.

Le format retenu est un `.js` (`window.DASHBOARD_DATA = {...}`) et non un `.json` : un navigateur refuse de
charger un JSON local via `fetch()` pour raison de securite (CORS sur `file://`), alors qu'une balise
`<script>` fonctionne. Le dashboard s'ouvre donc par un simple double-clic, sans serveur web.

In [42]:
exports_csv = {
    "kpi_panorama": pd.DataFrame([kpi]).T.rename(columns={0: "valeur"}),
    "serie_mensuelle": mensuel,
    "serie_journaliere": journalier,
    "causes_retard": causes_retard[["cause", "part_minutes_pct"]],
    "causes_annulation": causes_annul[["cause", "vols", "part_pct"]],
    "h1_fenetres_detail": h1_detail,
    "h1_effet_net_controle": h1_controle,
    "h2_compagnies": h2_compagnies,
    "h2_tranches_distance": h2_distance,
    "h3_routes_completes": routes,
    "h3_top_routes": TOP_ROUTES,
    "h3_routes_plus_risquees": routes_risque,
    "h3_top_aeroports": top_aeroports,
}
for nom, table in exports_csv.items():
    chemin = OUT / f"{nom}.csv"
    table.to_csv(chemin, index=(nom == "kpi_panorama"), encoding="utf-8-sig")
    print(f"{chemin.name:34s} {len(table):>7,} lignes")

kpi_panorama.csv                        11 lignes


serie_mensuelle.csv                     12 lignes
serie_journaliere.csv                  365 lignes
causes_retard.csv                        5 lignes
causes_annulation.csv                    4 lignes
h1_fenetres_detail.csv                   5 lignes
h1_effet_net_controle.csv                4 lignes
h2_compagnies.csv                       14 lignes
h2_tranches_distance.csv                 5 lignes
h3_routes_completes.csv              4,693 lignes
h3_top_routes.csv                       15 lignes
h3_routes_plus_risquees.csv             12 lignes
h3_top_aeroports.csv                    10 lignes


In [43]:
RESUME = {
    "genere_le": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "source": "Kaggle / US DOT - 2015 Flight Delays and Cancellations (vols domestiques US)",
    "perimetre": "Annee 2015 complete, 12 mois (H3 : 11 mois, cf. codes aeroport d'octobre)",
    "definitions": {
        "retard": f"arrivee avec >= {SEUIL_RETARD} min de retard, hors vols annules et deroutes",
        "vol_long": f"distance >= {SEUIL_LONG} miles (long-courrier domestique)",
        "vacances": {k: v for k, (_, v) in FENETRES.items()},
        "route": "couple oriente ORIGINE -> DESTINATION en codes IATA",
    },
    "panorama": kpi,
    "H1": {
        "statut": H1_STATUT, "resume": H1_RESUME,
        "taux_vacances_pct": round(TAUX_VAC, 2), "taux_hors_vacances_pct": round(TAUX_HORS, 2),
        "ecart_brut_points": round(ECART_NAIF, 2), "test_brut": H1_TEST,
        "poids_ete_dans_vacances_pct": round(POIDS_ETE, 1),
        "effet_net_par_fenetre": h1_controle.round(3).to_dict("records"),
    },
    "H2": {
        "statut": H2_STATUT, "resume": H2_RESUME,
        "correlations": H2_CORR, "test_groupes": H2_TEST,
        "taux_part_forte_pct": round(TAUX_FORT, 2), "taux_part_faible_pct": round(TAUX_FAIBLE, 2),
        "ecart_points": round(TAUX_FORT - TAUX_FAIBLE, 2),
        "mediane_part_vols_longs_pct": round(MEDIANE_PART, 2),
    },
    "H3": {
        "statut": H3_STATUT, "resume": H3_RESUME, "concentration": H3_CONC,
        "part_annulations_janvier_mars_pct": round(PART_HIVER, 1),
        "route_la_plus_annulee": {
            "route": routes.iloc[0].ROUTE, "annulations": int(routes.iloc[0].annulations),
            "vols": int(routes.iloc[0].vols), "taux_pct": round(float(routes.iloc[0].taux_annulation), 2),
        },
    },
    "duree_execution_sec": None,
}
(OUT / "resume_v2.json").write_text(json.dumps(RESUME, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: v for k, v in RESUME.items() if k in ("H1", "H2", "H3")},
                 indent=2, ensure_ascii=False)[:1400], "...")

{
  "H1": {
    "statut": "Confirmee, mais a preciser",
    "resume": "L'ecart brut est de +5.03 points (21.91 % contre 16.89 %), mais cet agregat est domine a 77 % par l'ete et masque des comportements opposes. Une fois chaque fenetre comparee a une reference comparable, l'effet le plus fort est Nouvel an (+18.25 points vs reste du meme mois), l'ete ne pese que +4.12 points, et Thanksgiving est negatif (-0.82 point). Ce ne sont donc pas 'les vacances' qui creent du retard, mais les pics courts de fin d'annee.",
    "taux_vacances_pct": 21.91,
    "taux_hors_vacances_pct": 16.89,
    "ecart_brut_points": 5.03,
    "test_brut": {
      "chi2": 21472.5,
      "p_value": 0.0,
      "dof": 1,
      "cramer_v": 0.0613,
      "risque_relatif": 1.298,
      "n": 5714008
    },
    "poids_ete_dans_vacances_pct": 77.1,
    "effet_net_par_fenetre": [
      {
        "fenetre": "Nouvel an",
        "nature": "effet fenetre",
        "reference": "reste du meme mois",
        "vols": 76931,
      

### 10.1 Carte : projection des routes en Python

Le dashboard doit s'ouvrir hors ligne, sans dependre d'un CDN. On fait donc en Python tout ce qu'une
librairie JavaScript de cartographie ferait dans le navigateur : lire le fond de carte des Etats, le
**projeter en Albers Equal Area** (projection standard pour les Etats-Unis, qui preserve les surfaces),
et exporter directement des traces SVG.

Le dashboard n'a alors plus qu'a afficher des chemins deja calcules : zero librairie externe, zero appel reseau.

In [44]:
FOND_CARTE = ASSETS / "states-10m.json"
URL_FOND = "https://cdn.jsdelivr.net/npm/us-atlas@3/states-10m.json"

if not FOND_CARTE.exists():
    import urllib.request
    print("Telechargement du fond de carte ...")
    FOND_CARTE.write_bytes(urllib.request.urlopen(URL_FOND, timeout=90).read())
topo = json.loads(FOND_CARTE.read_text(encoding="utf-8"))
print(f"Fond de carte : {FOND_CARTE.relative_to(RACINE)} ({FOND_CARTE.stat().st_size / 1024:.0f} Ko)")

Fond de carte : v2\assets\states-10m.json (112 Ko)


In [45]:
RAD = math.pi / 180

class AlbersConique:
    """Projection conique equivalente d'Albers (equal-area), parametree comme d3.geoConicEqualArea."""

    def __init__(self, paralleles, meridien, centre, echelle, translation):
        s0 = math.sin(paralleles[0] * RAD)
        self.n = (s0 + math.sin(paralleles[1] * RAD)) / 2
        self.c = 1 + s0 * (2 * self.n - s0)
        self.r0 = math.sqrt(self.c) / self.n
        self.meridien, self.k = meridien, echelle
        # Le centre est exprime dans le repere DEJA tourne : on ne lui reapplique pas le meridien.
        cx, cy = self._brut(centre[0], centre[1])
        self.dx = translation[0] - echelle * cx
        self.dy = translation[1] + echelle * cy

    def _brut(self, lon_tourne, lat):
        # Ramener la longitude dans [-180, 180] : sans cela les Aleoutiennes, qui franchissent
        # l'antimeridien (172 degres Est), seraient projetees a l'oppose de l'Alaska.
        lon_tourne = (lon_tourne + 180) % 360 - 180
        r = math.sqrt(self.c - 2 * self.n * math.sin(lat * RAD)) / self.n
        t = lon_tourne * RAD * self.n
        return r * math.sin(t), self.r0 - r * math.cos(t)

    def __call__(self, lon, lat):
        x, y = self._brut(lon + self.meridien, lat)
        return self.k * x + self.dx, self.dy - self.k * y


CARTE_L, CARTE_H = 860, 470

def albers_usa(largeur, hauteur):
    """Projection composite : 48 Etats contigus + encarts Alaska et Hawaii."""
    k, tx, ty = largeur * 1.12, largeur / 2, hauteur / 2
    return {
        "48": AlbersConique([29.5, 45.5], 96, (-0.6, 38.7), k, (tx, ty)),
        "AK": AlbersConique([55, 65], 154, (-2, 58.5), k * 0.32, (tx - 0.30 * k, ty + 0.20 * k)),
        "HI": AlbersConique([8, 18], 157, (-3, 19.9), k, (tx - 0.205 * k, ty + 0.212 * k)),
    }

PROJ = albers_usa(CARTE_L, CARTE_H)
ETATS_AK_HI = {"02": "AK", "15": "HI"}
TERRITOIRES = {"72", "78", "60", "66", "69"}     # Porto Rico, iles Vierges, Samoa, Guam, Mariannes

def projeter(code_etat, lon, lat):
    return PROJ[ETATS_AK_HI.get(code_etat, "48")](lon, lat)

def decoder_arcs(topo):
    """TopoJSON : les arcs sont quantifies et encodes en deltas. On les remet en lon/lat."""
    (sx, sy), (tx, ty) = topo["transform"]["scale"], topo["transform"]["translate"]
    arcs = []
    for arc in topo["arcs"]:
        x = y = 0
        points = []
        for dx, dy in arc:
            x += dx; y += dy
            points.append((x * sx + tx, y * sy + ty))
        arcs.append(points)
    return arcs

def assembler_anneau(arcs, indices):
    """Un anneau est une suite d'arcs ; un indice negatif ~i signifie l'arc i parcouru a l'envers."""
    points = []
    for i in indices:
        segment = arcs[~i][::-1] if i < 0 else arcs[i]
        points.extend(segment if not points else segment[1:])
    return points

arcs = decoder_arcs(topo)
TOLERANCE = 1.0      # decimation en pixels : allege le fichier sans effet visible

traces_etats = []
for geo in topo["objects"]["states"]["geometries"]:
    if geo["id"] in TERRITOIRES:
        continue
    anneaux = ([geo["arcs"]] if geo["type"] == "Polygon" else geo["arcs"])
    morceaux = []
    for polygone in anneaux:
        for anneau in polygone:
            pts = []
            for lon, lat in assembler_anneau(arcs, anneau):
                x, y = projeter(geo["id"], lon, lat)
                if not pts or abs(x - pts[-1][0]) + abs(y - pts[-1][1]) > TOLERANCE:
                    pts.append((x, y))
            if len(pts) >= 3:
                morceaux.append("M" + "L".join(f"{x:.1f},{y:.1f}" for x, y in pts) + "Z")
    if morceaux:
        traces_etats.append("".join(morceaux))

print(f"{len(traces_etats)} Etats projetes, "
      f"{sum(len(t) for t in traces_etats) / 1024:.0f} Ko de traces SVG")

51 Etats projetes, 93 Ko de traces SVG


In [46]:
# Etat de rattachement de chaque aeroport, pour choisir la bonne projection (encart AK / HI)
FIPS = {"AK": "02", "HI": "15"}
etat_aeroport = aeroports_ref.set_index("IATA_CODE").STATE.to_dict()

def position(code_iata):
    if code_iata not in coords.index:
        return None
    lat, lon = coords.loc[code_iata, "LATITUDE"], coords.loc[code_iata, "LONGITUDE"]
    if pd.isna(lat) or pd.isna(lon):
        return None
    return projeter(FIPS.get(etat_aeroport.get(code_iata), "48"), float(lon), float(lat))

def arc_svg(p1, p2):
    """Arc de cercle stylise entre deux aeroports (courbe de Bezier quadratique)."""
    (x1, y1), (x2, y2) = p1, p2
    longueur = math.hypot(x2 - x1, y2 - y1)
    courbure = min(70, max(14, longueur * 0.19))
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2 - courbure
    return f"M{x1:.1f},{y1:.1f}Q{mx:.1f},{my:.1f} {x2:.1f},{y2:.1f}"

N_ROUTES_CARTE = 180
traces_routes, ignorees = [], 0
for _, r in routes.head(N_ROUTES_CARTE).iterrows():
    a, b = position(r.origine), position(r.destination)
    if a is None or b is None:
        ignorees += 1
        continue
    traces_routes.append({"route": r.ROUTE, "d": arc_svg(a, b),
                          "annulations": int(r.annulations), "vols": int(r.vols),
                          "taux": round(float(r.taux_annulation), 2), "rang": int(r.rang)})

codes_affiches = sorted({c for r in routes.head(10).itertuples()
                         for c in (r.origine, r.destination)})
marqueurs = []
for c in codes_affiches:
    p = position(c)
    if p:
        ligne = aeroports_annul[aeroports_annul.aeroport.eq(c)]
        marqueurs.append({"code": c, "x": round(p[0], 1), "y": round(p[1], 1),
                          "ville": coords.CITY.get(c, ""),
                          "annulations": int(ligne.annulations.iloc[0]) if len(ligne) else 0,
                          "taux": round(float(ligne.taux_annulation.iloc[0]), 2) if len(ligne) else 0.0})

print(f"{len(traces_routes)} routes tracees ({ignorees} ignorees faute de coordonnees), "
      f"{len(marqueurs)} aeroports marques")

180 routes tracees (0 ignorees faute de coordonnees), 6 aeroports marques


In [47]:
def alleger(xs, ys, n=200):
    """Sous-echantillonne une courbe pour le dashboard, en gardant les extremites."""
    idx = np.unique(np.linspace(0, len(xs) - 1, n).astype(int))
    return [[round(float(xs[i]), 3), round(float(ys[i]), 3)] for i in idx]

DONNEES_DASHBOARD = {
    "meta": {
        "genere_le": RESUME["genere_le"], "source": RESUME["source"],
        "perimetre": RESUME["perimetre"],
        "definitions": {"retard": RESUME["definitions"]["retard"],
                        "vol_long": RESUME["definitions"]["vol_long"],
                        "route": RESUME["definitions"]["route"]},
    },
    "kpi": kpi,
    "reperes": {"taux_retard_global": round(TAUX_RETARD_GLOBAL, 2),
                "taux_annulation_global": round(TAUX_ANNUL_GLOBAL, 2)},
    "mensuel": mensuel.round(2).to_dict("records"),
    "journalier": [{"j": int(r.jour_annee), "date": r.DATE.strftime("%d/%m"),
                    "retard": round(float(r.taux_retard), 2),
                    "annulation": round(float(r.taux_annulation), 2)}
                   for r in journalier.itertuples()],
    "fenetres": [{"nom": n, "debut": int(vols.loc[m, "JOUR_ANNEE"].min()),
                  "fin": int(vols.loc[m, "JOUR_ANNEE"].max()), "dates": lib}
                 for n, (m, lib) in FENETRES.items()],
    "causes_retard": causes_retard[["cause", "part_minutes_pct"]].round(2).to_dict("records"),
    "causes_annulation": causes_annul[["cause", "vols", "part_pct"]].round(2).to_dict("records"),
    "h1": {
        "statut": H1_STATUT, "resume": H1_RESUME,
        "taux_vacances": round(TAUX_VAC, 2), "taux_hors_vacances": round(TAUX_HORS, 2),
        "ecart_brut": round(ECART_NAIF, 2), "cramer_v": H1_TEST["cramer_v"],
        "risque_relatif": H1_TEST["risque_relatif"], "poids_ete": round(POIDS_ETE, 1),
        "detail": h1_detail.round(2).to_dict("records"),
        "controle": h1_controle.round(2).to_dict("records"),
        "pires_journees": [{"date": r.DATE.strftime("%d/%m"), "retard": round(float(r.taux_retard), 1),
                            "vols": int(r.vols)} for r in PIRES_RETARDS.itertuples()],
    },
    "h2": {
        "statut": H2_STATUT, "resume": H2_RESUME, "correlations": H2_CORR,
        "regression": {"pente": round(float(reg.slope), 4), "ordonnee": round(float(reg.intercept), 3)},
        "taux_part_forte": round(TAUX_FORT, 2), "taux_part_faible": round(TAUX_FAIBLE, 2),
        "mediane_part": round(MEDIANE_PART, 2),
        "compagnies": [{"code": r.AIRLINE, "nom": r.compagnie, "vols": int(r.vols),
                        "part_longs": round(float(r.part_vols_longs), 2),
                        "taux_retard": round(float(r.taux_retard), 2),
                        "retard_moyen": round(float(r.retard_moyen), 2),
                        "distance_moyenne": round(float(r.distance_moyenne))}
                       for r in h2_compagnies.itertuples()],
        "tranches": [{"tranche": str(r.tranche), "vols": int(r.vols),
                      "taux_retard": round(float(r.taux_retard), 2),
                      "retard_moyen": round(float(r.retard_moyen), 2),
                      "rattrapage": round(float(r.rattrapage_min), 2)}
                     for r in h2_distance.itertuples()],
    },
    "h3": {
        "statut": H3_STATUT, "resume": H3_RESUME, "concentration": H3_CONC,
        "part_hiver": round(PART_HIVER, 1),
        "lorenz_annulations": alleger(part_routes, cum_annulations),
        "lorenz_vols": alleger(part_routes, cum_vols),
        "top_routes": [{"rang": int(r.rang), "route": r.ROUTE,
                        "origine": r.origine, "destination": r.destination,
                        "ville_origine": r.ville_origine, "ville_destination": r.ville_destination,
                        "vols": int(r.vols), "annulations": int(r.annulations),
                        "taux": round(float(r.taux_annulation), 2)}
                       for r in TOP_ROUTES.head(10).itertuples()],
        "routes_risquees": [{"route": r.ROUTE, "vols": int(r.vols), "annulations": int(r.annulations),
                             "taux": round(float(r.taux_annulation), 2)}
                            for r in routes_risque.head(8).itertuples()],
        "top_aeroports": [{"code": r.aeroport, "vols": int(r.vols), "annulations": int(r.annulations),
                           "taux": round(float(r.taux_annulation), 2)}
                          for r in top_aeroports.itertuples()],
        # Octobre est hors perimetre (codes aeroport numeriques) : on renvoie explicitement
        # une valeur nulle plutot qu'un mois absent, pour que le dashboard puisse le signaler.
        "saison_prioritaires": [{"mois": m, "taux": (round(float(saison["taux"].loc[m]), 2)
                                                     if m in saison.index else None)}
                                for m in range(1, 13)],
    },
    "carte": {"largeur": CARTE_L, "hauteur": CARTE_H,
              "etats": traces_etats, "routes": traces_routes, "aeroports": marqueurs},
}

DUREE = round(time.time() - T0, 1)
DONNEES_DASHBOARD["meta"]["duree_execution_sec"] = DUREE
RESUME["duree_execution_sec"] = DUREE
(OUT / "resume_v2.json").write_text(json.dumps(RESUME, indent=2, ensure_ascii=False), encoding="utf-8")

cible = DASH / "dashboard_data.js"
cible.write_text(
    "// Genere automatiquement par v2/notebook_analyse_v2.ipynb - NE PAS EDITER A LA MAIN\n"
    f"// Genere le {RESUME['genere_le']}\n"
    "window.DASHBOARD_DATA = "
    + json.dumps(DONNEES_DASHBOARD, ensure_ascii=False, separators=(",", ":"))
    + ";\n", encoding="utf-8")

print(f"{cible.relative_to(RACINE)}  ({cible.stat().st_size / 1024:.0f} Ko)")
print(f"Duree totale d'execution : {DUREE} s")

v2\dashboard\dashboard_data.js  (154 Ko)
Duree totale d'execution : 27.4 s


---
## 11. Synthese

Le tableau ci-dessous est **genere a partir des variables calculees** : il ne peut pas diverger des resultats.

In [48]:
from IPython.display import Markdown

lignes = [
    "| Hypothese | Verdict | Ce que disent les donnees |",
    "|---|---|---|",
    f"| **H1** — plus de retards pendant les vacances | **{H1_STATUT}** | {H1_RESUME} |",
    f"| **H2** — les compagnies long-courrier ont plus de retards | **{H2_STATUT}** | {H2_RESUME} |",
    f"| **H3** — une minorite de routes concentre les annulations | **{H3_STATUT}** | {H3_RESUME} |",
]

conclusion = f"""
### Conclusion generale

Sur {nb_fr(kpi['vols_programmes'])} vols programmes en 2015, {kpi['taux_retard_pct']} % arrivent avec au moins
{SEUIL_RETARD} minutes de retard et {kpi['taux_annulation_pct']} % sont annules.

Les trois hypotheses ne se valident pas de la meme facon, et c'est le principal enseignement de l'analyse :

1. **H1 se confirme, mais pas pour la raison attendue.** Ce ne sont pas « les vacances » qui produisent du
   retard, mais deux pics courts de fin d'annee. Thanksgiving, pourtant le plus gros week-end de deplacement
   du pays, n'a aucun effet ({float(h1_controle.loc[h1_controle.fenetre.eq('Thanksgiving'), 'effet_net_points'].iloc[0]):+.2f} point).

2. **H2 ne se valide pas.** La structure de reseau d'une compagnie n'explique pas sa ponctualite
   (R2 = {H2_CORR['r2']:.3f} sur {H2_CORR['n_compagnies']} compagnies). La distance a meme un effet protecteur :
   les vols longs rattrapent du retard en vol.

3. **H3 se confirme, a condition de la mesurer correctement.** La concentration brute est en grande partie
   mecanique ; le resultat solide est le sur-risque a volume comparable (x{SURRISQUE:.2f}), porte par les
   navettes courtes du Nord-Est en hiver.

**Fil conducteur** : retards et annulations repondent a deux logiques differentes. Les retards suivent le
**calendrier** (pics de trafic de fin d'annee), les annulations suivent la **meteo et la topologie du reseau**
(hiver, Nord-Est, rotations courtes). Un pilotage operationnel efficace doit donc les traiter separement —
c'est la lecture que propose le dashboard.
"""

Markdown("\n".join(lignes) + "\n" + conclusion)

| Hypothese | Verdict | Ce que disent les donnees |
|---|---|---|
| **H1** — plus de retards pendant les vacances | **Confirmee, mais a preciser** | L'ecart brut est de +5.03 points (21.91 % contre 16.89 %), mais cet agregat est domine a 77 % par l'ete et masque des comportements opposes. Une fois chaque fenetre comparee a une reference comparable, l'effet le plus fort est Nouvel an (+18.25 points vs reste du meme mois), l'ete ne pese que +4.12 points, et Thanksgiving est negatif (-0.82 point). Ce ne sont donc pas 'les vacances' qui creent du retard, mais les pics courts de fin d'annee. |
| **H2** — les compagnies long-courrier ont plus de retards | **Non validee** | Sur les 14 compagnies, la correlation entre part de vols longs et taux de retard est de -0.179 (Pearson, p = 0.54) et -0.222 (Spearman, p = 0.45) : elle est faible, de signe contraire a l'hypothese, et non significative. La part de vols longs n'explique que 3.2 % des ecarts de ponctualite entre compagnies. La comparaison des deux groupes donne 18.81 % contre 18.51 %, soit +0.30 point, un ecart sans portee pratique. Au niveau du vol, la distance ne degrade pas la ponctualite : les vols les plus longs rattrapent en moyenne 8.9 minutes en vol contre 2.9 pour les plus courts. |
| **H3** — une minorite de routes concentre les annulations | **Confirmee, avec une reserve importante** | 455 routes sur 4 693 (9.70 %) portent la moitie des 87 430 annulations du perimetre, et le Gini des annulations (0.683) depasse celui du trafic (0.558). La concentration est donc reelle, mais elle est pour une bonne part mecanique : les 10 % de routes en tete portent 50.9 % des annulations et deja 38.7 % des vols. Le resultat solide est le sur-risque a volume comparable : 2.92 % d'annulation sur les routes prioritaires contre 1.14 % ailleurs, soit x2.56. Ces routes sont majoritairement des navettes courtes du Nord-Est, et 52 % de leurs annulations tombent au premier trimestre. |

### Conclusion generale

Sur 5 819 079 vols programmes en 2015, 18.61 % arrivent avec au moins
15 minutes de retard et 1.54 % sont annules.

Les trois hypotheses ne se valident pas de la meme facon, et c'est le principal enseignement de l'analyse :

1. **H1 se confirme, mais pas pour la raison attendue.** Ce ne sont pas « les vacances » qui produisent du
   retard, mais deux pics courts de fin d'annee. Thanksgiving, pourtant le plus gros week-end de deplacement
   du pays, n'a aucun effet (-0.82 point).

2. **H2 ne se valide pas.** La structure de reseau d'une compagnie n'explique pas sa ponctualite
   (R2 = 0.032 sur 14 compagnies). La distance a meme un effet protecteur :
   les vols longs rattrapent du retard en vol.

3. **H3 se confirme, a condition de la mesurer correctement.** La concentration brute est en grande partie
   mecanique ; le resultat solide est le sur-risque a volume comparable (x2.56), porte par les
   navettes courtes du Nord-Est en hiver.

**Fil conducteur** : retards et annulations repondent a deux logiques differentes. Les retards suivent le
**calendrier** (pics de trafic de fin d'annee), les annulations suivent la **meteo et la topologie du reseau**
(hiver, Nord-Est, rotations courtes). Un pilotage operationnel efficace doit donc les traiter separement —
c'est la lecture que propose le dashboard.


---
## 12. Fichiers produits

Tout ce qui suit est regenere a chaque execution complete du notebook. Aucun autre script n'est necessaire.

**`v2/outputs/` — tableaux**

| Fichier | Contenu |
|---|---|
| `kpi_panorama.csv` | indicateurs generaux du dataset |
| `serie_mensuelle.csv`, `serie_journaliere.csv` | taux de retard et d'annulation dans le temps |
| `causes_retard.csv`, `causes_annulation.csv` | decomposition des causes |
| `h1_fenetres_detail.csv` | taux de retard par fenetre de mobilite |
| `h1_effet_net_controle.csv` | effet net de chaque fenetre vs sa reference |
| `h2_compagnies.csv` | les 14 compagnies : part de vols longs et ponctualite |
| `h2_tranches_distance.csv` | retard et rattrapage par tranche de distance |
| `h3_routes_completes.csv` | les 4 693 routes, avec cumuls de concentration |
| `h3_top_routes.csv`, `h3_routes_plus_risquees.csv`, `h3_top_aeroports.csv` | classements |
| `resume_v2.json` | synthese chiffree complete de l'analyse |

**`v2/outputs/figures/` — 6 graphiques PNG** pour le rapport et la presentation.

**`v2/dashboard/`**

| Fichier | Role |
|---|---|
| `dashboard_data.js` | genere ici : toutes les donnees affichees par le dashboard |
| `index.html` | le dashboard, a ouvrir par double-clic (aucun serveur, aucune connexion requise) |

**`v2/assets/states-10m.json`** — fond de carte des Etats-Unis, telecharge une seule fois puis mis en cache.